# 02 — ResGCN E0–E7: Signer-Independent LOSO Training

Notebook ini menjalankan eksperimen faktorial **E0–E7** untuk pengenalan kata BISINDO berbasis pose menggunakan **ResGCN_N39_R4**.

Pipeline penelitian:

1. raw pose `(T, 65, 2)` hasil ekstraksi;
2. pembagian signer-independent dengan skema Leave-One-Signer-Out (LOSO);
3. 15% kandidat data latih dipisahkan sebagai validasi internal secara label-stratified;
4. tiga faktor eksperimen: interpolasi linear, normalisasi spasial, dan augmentasi pose;
5. standardisasi temporal seluruh sequence menjadi **64 frame**;
6. input model berbentuk `(N, 2, 64, 65)`;
7. augmentasi hanya diterapkan pada `train_fit`;
8. model terbaik dipilih berdasarkan `validation loss`;
9. evaluasi akhir dilakukan pada unseen test signer.

Evaluasi LOSO utama menggunakan signer 1–4 sebagai test signer. Signer 0 tetap berada pada training pool karena cakupan labelnya tidak memenuhi seluruh 32 kelas untuk digunakan sebagai test fold.

Hasil final penelitian menggunakan tiga random seed: **42, 123, dan 456**. Variabel `RUN_SEED` menentukan seed aktif dan output setiap seed disimpan pada direktori terpisah.

## Dataset

Penelitian ini menggunakan **WL-BISINDO (Word-Level BISINDO)** yang diperkenalkan oleh **Grace Oktaviani Kindy, Glenn Leonali, dan Henry Lucky** dari BINUS University dalam paper:

> *Word-Level BISINDO: A Novel Video Indonesian Sign Language Dataset and Baseline Methods*  
> Procedia Computer Science, Vol. 269, 2025, pp. 249–258  
> DOI: https://doi.org/10.1016/j.procs.2025.08.277

WL-BISINDO merupakan dataset video **Bahasa Isyarat Indonesia (BISINDO)** pada tingkat kata dengan fokus pada **varian regional Banten**. Dataset berisi **1.600 video RGB**, **32 isolated signs**, dan **5 signers**.

Dataset tersedia secara publik melalui Kaggle:

https://www.kaggle.com/datasets/glennleonali/wl-bisindo

Lisensi dataset: **CC BY-NC 4.0** (penggunaan non-komersial dengan atribusi).

## Menjalankan Eksperimen

Untuk penggunaan Google Colab, satu eksperimen (`E0`, `E1`, …, `E7`) per sesi biasanya lebih aman. Untuk menjalankan seluruh konfigurasi dalam satu seed, isi `ACTIVE_EXPERIMENTS` dengan seluruh E0–E7.


## 1. Konfigurasi Run

Ubah terutama `RUN_SEED` dan `ACTIVE_EXPERIMENTS`. Nilai lain dipertahankan sesuai pipeline eksperimen final.


In [ ]:
# ============================================================
# 1. KONFIGURASI RUN
# ============================================================

# Seed yang dilaporkan pada penelitian final.
FINAL_STUDY_SEEDS = [42, 123, 456]
RUN_SEED = 42

if not isinstance(RUN_SEED, int):
    raise TypeError("RUN_SEED harus berupa integer.")

if RUN_SEED not in FINAL_STUDY_SEEDS:
    print(
        f"[WARN] RUN_SEED={RUN_SEED} bukan salah satu seed final "
        f"{FINAL_STUDY_SEEDS}. Run tetap diperbolehkan, tetapi bukan bagian "
        "dari tiga pengulangan yang dilaporkan."
    )

# Disarankan satu eksperimen per sesi Colab.
# Untuk full reproduction satu seed:
# ACTIVE_EXPERIMENTS = ["E0","E1","E2","E3","E4","E5","E6","E7"]
ACTIVE_EXPERIMENTS = ["E0"]

# Resume-safe defaults.
FORCE_REBUILD_DATA = False
FORCE_RERUN_TRAINING = False

AUTO_BUILD_DATA = True
AUTO_TRAIN = True
AUTO_SUMMARIZE_EXPERIMENT = True
AUTO_TRY_GLOBAL_SUMMARY = True

# QC visual tidak diperlukan untuk setiap rerun dan tidak memengaruhi training.
AUTO_SAVE_QC_VISUAL = False

print("RUN_SEED:", RUN_SEED)
print("ACTIVE_EXPERIMENTS:", ACTIVE_EXPERIMENTS)
print("FORCE_REBUILD_DATA:", FORCE_REBUILD_DATA)
print("FORCE_RERUN_TRAINING:", FORCE_RERUN_TRAINING)
print("AUTO_BUILD_DATA:", AUTO_BUILD_DATA)
print("AUTO_TRAIN:", AUTO_TRAIN)
print("AUTO_SAVE_QC_VISUAL:", AUTO_SAVE_QC_VISUAL)


## 2. Mount Drive dan Definisi Path

Input raw pose tetap dibaca dari `data/data_pose_raw/`. Artifact preprocessing dan hasil training disimpan dalam folder terpisah untuk setiap seed agar repeated run tidak saling menimpa.


In [ ]:
# ============================================================
# 2. MOUNT DRIVE DAN DEFINISI PATH
# ============================================================

from google.colab import drive
from pathlib import Path
import os, re, json, time, math, random, shutil, subprocess, tempfile
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm.auto import tqdm
from IPython.display import display

drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Skripsi/model")
DATA_DIR = ROOT / "data"

# Input dari tahap ekstraksi pose dan metadata.
DRIVE_RAW_TAR = DATA_DIR / "data_pose_raw.tar.gz"
DRIVE_RAW_DIR = DATA_DIR / "data_pose_raw"
DRIVE_SPLIT_DIR = DATA_DIR / "splits"

# Output dipisahkan berdasarkan seed.
SEED_TAG = f"seed_{RUN_SEED}"
EXPERIMENT_DATA_ROOT = DATA_DIR / "experiments_e0_e7_multiseed" / SEED_TAG
RUN_ROOT = ROOT / "runs" / "loso_e0_e7_multiseed" / SEED_TAG

# Workspace lokal Colab untuk mengurangi small-file I/O ke Drive.
VM = Path(f"/content/workdir_e0_e7_{SEED_TAG}")
VM_RAW_DIR = VM / "data_pose_raw"
VM_SPLIT_DIR = VM / "splits"

for p in [EXPERIMENT_DATA_ROOT, RUN_ROOT, VM, VM_RAW_DIR, VM_SPLIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT                 :", ROOT)
print("SEED_TAG             :", SEED_TAG)
print("EXPERIMENT_DATA_ROOT :", EXPERIMENT_DATA_ROOT)
print("RUN_ROOT             :", RUN_ROOT)
print("RAW tar exists       :", DRIVE_RAW_TAR.exists())
print("RAW dir exists       :", DRIVE_RAW_DIR.exists())
print("Split dir exists     :", DRIVE_SPLIT_DIR.exists())


## 3. Reproducibility, Device, dan Hyperparameter

`RUN_SEED` mengontrol inisialisasi model, internal validation split, urutan training, dan augmentasi acak. Default `STRICT_DETERMINISM=False` digunakan untuk training normal; GPU/CUDA dapat tetap memiliki variasi numerik kecil antar-runtime.


In [ ]:
# ============================================================
# 3. REPRODUCIBILITY, DEVICE, DAN HYPERPARAMETER
# ============================================================

import sys
import platform
import importlib.metadata as importlib_metadata

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

SEED = RUN_SEED
TARGET_T = 64
N_JOINTS = 65
N_COORDS = 2
NUM_CLASSES = 32
TEST_SIGNERS = [1, 2, 3, 4]
VAL_FRAC = 0.15

# False = perilaku eksperimen asli (lebih cepat; tidak menjamin bitwise identity).
# True  = mode lebih ketat untuk debugging/reproduction, tetapi dapat lebih lambat.
STRICT_DETERMINISM = False

HYPERPARAMS = {
    "epochs": 350,
    "batch_size": 32,
    "lr": 0.01,
    "save_every": 5,
    "early_stop": True,
    "warmup_epochs": 25,
    "patience": 60,
    "min_delta": 1e-4,
    "optimizer": "Adam",
    "scheduler": "CyclicLR",
    "selection_metric": "val_loss",
    "model": "ResGCN_N39_R4",
    "R": 4,
    "t_kernel": 9,
    "dropout": 0.1,
}

AUGMENT_CONFIG = {
    "name": "pose_spatial_aug_v1",
    "active_split": "train_fit_only",
    "input_shape": "(C,T,V)=(2,64,65)",
    "order": ["small_rotation", "spatial_jitter"],
    "rotation_degrees": [-10, 10],
    "rotation_probability": 0.5,
    "rotation_center": "valid_joint_centroid",
    "jitter_mean": 0.0,
    "jitter_sigma": 0.015,
    "jitter_probability": 0.5,
}

# Joint index sesuai joints65.yaml
# LH: 0-20 | RH: 21-41 | Head: 42-52
# Body: 53=NECK, 54=L_SH, 55=R_SH, 56=L_EL, 57=R_EL,
#       58=L_WR, 59=R_WR, 60=L_HIP, 61=R_HIP, 62=MID_HIP
IDX_L_SHOULDER = 54
IDX_R_SHOULDER = 55
IDX_L_HIP = 60
IDX_R_HIP = 61
IDX_MID_HIP = 62

def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Konfigurasi default untuk training normal.
    torch.backends.cudnn.deterministic = bool(STRICT_DETERMINISM)
    torch.backends.cudnn.benchmark = not bool(STRICT_DETERMINISM)

seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_TYPE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None

environment = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "seed": int(SEED),
    "python": sys.version,
    "platform": platform.platform(),
    "device": DEVICE,
    "gpu_type": GPU_TYPE,
    "strict_determinism": bool(STRICT_DETERMINISM),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scipy": package_version("scipy"),
        "torch": package_version("torch"),
        "scikit-learn": package_version("scikit-learn"),
        "tqdm": package_version("tqdm"),
    },
}

with open(RUN_ROOT / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

with open(RUN_ROOT / "study_run_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "seed": int(SEED),
        "active_experiments": ACTIVE_EXPERIMENTS,
        "target_T": TARGET_T,
        "n_joints": N_JOINTS,
        "n_coords": N_COORDS,
        "num_classes": NUM_CLASSES,
        "test_signers": TEST_SIGNERS,
        "val_frac": VAL_FRAC,
        "hyperparams": HYPERPARAMS,
        "augment_config": AUGMENT_CONFIG,
    }, f, indent=2)

print("SEED:", SEED)
print("DEVICE:", DEVICE)
print("GPU_TYPE:", GPU_TYPE)
print("STRICT_DETERMINISM:", STRICT_DETERMINISM)
display(pd.DataFrame([HYPERPARAMS]))


## 4. Load Metadata Dataset

Raw pose dan metadata split disalin ke workspace lokal Colab. Cell ini juga memvalidasi identitas signer dan label sebelum eksperimen dijalankan.


In [ ]:
# ============================================================
# 4. LOAD METADATA DATASET
# ============================================================

pat = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)", re.IGNORECASE)


def ensure_raw_and_split_available():
    """Copy/extract raw pose dan split CSV ke VM lokal Colab."""
    # Catatan paralel:
    # Jika archive raw belum tersedia, jalankan satu sesi Colab terlebih dahulu sampai
    # data_pose_raw.tar.gz terbentuk. Setelah archive tersedia di Drive, eksperimen
    # boleh dijalankan paralel dari beberapa akun/sesi.
    print("Catatan paralel: jika data_pose_raw.tar.gz belum ada, jalankan satu sesi ini sendiri dulu. Jangan paralel sebelum archive raw terbentuk.")
    if not DRIVE_RAW_TAR.exists() and DRIVE_RAW_DIR.exists():
        print("[WARNING] data_pose_raw.tar.gz belum ada. Notebook akan membuat archive dari satu sesi ini.")

    # Raw pose
    n_existing = len(list(VM_RAW_DIR.glob("*.npz")))
    if n_existing > 0:
        print(f"VM sudah memiliki {n_existing} file raw — skip extract")
    else:
        if DRIVE_RAW_TAR.exists():
            vm_tar = VM / "data_pose_raw.tar.gz"
            print("Copy archive raw ke VM...")
            shutil.copy2(str(DRIVE_RAW_TAR), str(vm_tar))
            print("Extract raw ke VM...")
            result = subprocess.run(["tar", "-xzf", str(vm_tar), "-C", str(VM)], capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(result.stderr)
            vm_tar.unlink(missing_ok=True)
        elif DRIVE_RAW_DIR.exists():
            print("Archive raw tidak ditemukan. Membuat archive dari data_pose_raw di Drive lalu extract ke VM...")
            result = subprocess.run(
                ["tar", "-czf", str(DRIVE_RAW_TAR), "-C", str(DATA_DIR), "data_pose_raw"],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr)
            vm_tar = VM / "data_pose_raw.tar.gz"
            shutil.copy2(str(DRIVE_RAW_TAR), str(vm_tar))
            result = subprocess.run(["tar", "-xzf", str(vm_tar), "-C", str(VM)], capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(result.stderr)
            vm_tar.unlink(missing_ok=True)
        else:
            raise FileNotFoundError("Tidak menemukan data_pose_raw.tar.gz atau folder data_pose_raw di Drive.")

        n_after = len(list(VM_RAW_DIR.glob("*.npz")))
        print(f"Raw siap di VM: {n_after} file")

    # Split CSV
    copied = 0
    if not DRIVE_SPLIT_DIR.exists():
        raise FileNotFoundError(f"Folder split tidak ditemukan: {DRIVE_SPLIT_DIR}")
    for f in DRIVE_SPLIT_DIR.rglob("*.csv"):
        dest = VM_SPLIT_DIR / f.name
        if not dest.exists():
            shutil.copy2(str(f), str(dest))
            copied += 1
    print(f"Split CSV siap di VM: {len(list(VM_SPLIT_DIR.glob('*.csv')))} file | copied={copied}")


def parse_video_id(vid):
    m = pat.match(str(vid))
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))


def load_metadata_from_split_csv():
    csvs = sorted(VM_SPLIT_DIR.glob("*.csv"))
    dfs = []
    for p in csvs:
        d = pd.read_csv(p)
        if "video_id" in d.columns:
            dfs.append(d[["video_id"]])

    if not dfs:
        raise RuntimeError("Tidak ada CSV split dengan kolom video_id di VM_SPLIT_DIR.")

    all_ids = pd.concat(dfs)["video_id"].dropna().astype(str).unique().tolist()
    rows = []
    for vid in sorted(all_ids):
        parsed = parse_video_id(vid)
        if parsed:
            s, lab, samp = parsed
            rows.append({"video_id": vid, "signer_id": s, "label": lab, "sample_id": samp})

    df_meta = pd.DataFrame(rows).sort_values(["signer_id", "label", "sample_id"]).reset_index(drop=True)
    if df_meta.empty:
        raise RuntimeError("Metadata kosong. Cek pola video_id: signer{n}_label{k}_sample{i}.")
    return df_meta

ensure_raw_and_split_available()
df_meta = load_metadata_from_split_csv()

# Validasi ringan metadata. Tujuannya bukan mengubah data, tetapi memberi peringatan
# jika metadata berubah sebelum eksperimen final dijalankan.
ALL_SIGNERS = sorted(df_meta["signer_id"].unique().tolist())
missing_test_signers = set(TEST_SIGNERS) - set(ALL_SIGNERS)
if missing_test_signers:
    raise ValueError(f"TEST_SIGNERS tidak ditemukan di metadata: {missing_test_signers}")

label_min = int(df_meta["label"].min())
label_max = int(df_meta["label"].max())
n_labels = int(df_meta["label"].nunique())


# Dataset-level sanity check sesuai data penelitian.
if len(df_meta) != 1600:
    raise ValueError(f"Jumlah video tidak sesuai ekspektasi penelitian: {len(df_meta)} != 1600")
if set(ALL_SIGNERS) != {0, 1, 2, 3, 4}:
    raise ValueError(f"Signer tidak sesuai ekspektasi penelitian: {ALL_SIGNERS}")

print(f"Total video: {len(df_meta)}")
print("ALL_SIGNERS:", ALL_SIGNERS)
print("TEST_SIGNERS:", TEST_SIGNERS)
print("Label min:", label_min)
print("Label max:", label_max)
print("Jumlah label unik:", n_labels)

if label_min != 0 or label_max != NUM_CLASSES - 1 or n_labels != NUM_CLASSES:
    raise ValueError(
        f"Label tidak sesuai ekspektasi 0-{NUM_CLASSES-1}. "
        f"Ditemukan min={label_min}, max={label_max}, n_labels={n_labels}."
    )

print("Distribusi signer:")
display(df_meta.groupby("signer_id")["video_id"].count().rename("n_videos").reset_index())


## 5. Registry Eksperimen E0–E7

Delapan konfigurasi merupakan desain faktorial 2³ dari tiga faktor: interpolasi (`interp`), normalisasi spasial (`norm`), dan augmentasi (`augment`).


In [ ]:

# ============================================================
# 5. REGISTRY EKSPERIMEN E0-E7
# ============================================================

# Desain eksperimen mengikuti kombinasi faktorial dari tiga komponen:
# I = interpolasi missing keypoint
# N = normalisasi geometrik/spasial
# A = augmentasi data latih
EXPERIMENTS = {
    "E0": {"name": "baseline_minimum", "interp": False, "norm": False, "augment": False},
    "E1": {"name": "interp_only", "interp": True, "norm": False, "augment": False},
    "E2": {"name": "norm_only", "interp": False, "norm": True, "augment": False},
    "E3": {"name": "aug_only", "interp": False, "norm": False, "augment": True},
    "E4": {"name": "interp_norm", "interp": True, "norm": True, "augment": False},
    "E5": {"name": "interp_aug", "interp": True, "norm": False, "augment": True},
    "E6": {"name": "norm_aug", "interp": False, "norm": True, "augment": True},
    "E7": {"name": "interp_norm_aug", "interp": True, "norm": True, "augment": True},
}

for exp_id in ACTIVE_EXPERIMENTS:
    if exp_id not in EXPERIMENTS:
        raise ValueError(f"Experiment tidak dikenal: {exp_id}. Pilih salah satu dari {list(EXPERIMENTS)}")

df_exp = pd.DataFrame([{"exp_id": k, **v} for k, v in EXPERIMENTS.items()])
display(df_exp)


## 6. Split LOSO + Internal Validation

Untuk setiap fold, satu signer (`1–4`) menjadi test signer dan tidak boleh muncul pada `train_fit` maupun `val_internal`. Sebanyak 15% kandidat data latih dipisahkan sebagai validasi internal secara label-stratified.


In [ ]:

# ============================================================
# 6. FUNGSI SPLIT LOSO + INTERNAL VALIDATION
# ============================================================


def get_loso_candidate_and_test(test_signer):
    """LOSO utama: test signer tidak pernah masuk train_fit atau val_internal."""
    train_candidate_signers = [s for s in ALL_SIGNERS if s != test_signer]
    return train_candidate_signers, test_signer


def stratified_internal_split(df_candidate, val_frac=VAL_FRAC, seed=SEED):
    """
    Membagi kandidat training menjadi train_fit dan val_internal.
    Stratifikasi dilakukan per label agar validation tetap mewakili seluruh kelas.
    """
    train_parts, val_parts = [], []
    for lab, g in df_candidate.groupby("label", sort=True):
        g = g.sample(frac=1.0, random_state=seed + int(lab)).reset_index(drop=True)
        if len(g) <= 1:
            n_val = 0
        else:
            n_val = max(1, int(round(len(g) * val_frac)))
            n_val = min(n_val, len(g) - 1)
        val_parts.append(g.iloc[:n_val])
        train_parts.append(g.iloc[n_val:])

    df_train_fit = pd.concat(train_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_val_int = pd.concat(val_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_train_fit, df_val_int


def make_fold_split(df_meta, fold):
    candidate_signers, test_signer = get_loso_candidate_and_test(fold)
    df_candidate = df_meta[df_meta["signer_id"].isin(candidate_signers)].copy()
    df_test = df_meta[df_meta["signer_id"] == test_signer].copy()
    df_train_fit, df_val_int = stratified_internal_split(df_candidate, val_frac=VAL_FRAC, seed=SEED + fold)
    return df_train_fit, df_val_int, df_test


def assert_no_leakage(df_train_fit, df_val_int, df_test, test_signer):
    assert test_signer not in set(df_train_fit["signer_id"]), "Leakage: test signer masuk train_fit"
    assert test_signer not in set(df_val_int["signer_id"]), "Leakage: test signer masuk val_internal"
    assert set(df_train_fit["video_id"]).isdisjoint(set(df_val_int["video_id"])), "Train/val overlap"
    assert set(df_train_fit["video_id"]).isdisjoint(set(df_test["video_id"])), "Train/test overlap"
    assert set(df_val_int["video_id"]).isdisjoint(set(df_test["video_id"])), "Val/test overlap"


def summarize_split(df, name):
    return {
        "subset": name,
        "n": int(len(df)),
        "n_labels": int(df["label"].nunique()) if len(df) else 0,
        "signers": ",".join(map(str, sorted(df["signer_id"].unique()))) if len(df) else "",
        "min_per_label": int(df.groupby("label")["video_id"].count().min()) if len(df) else 0,
        "max_per_label": int(df.groupby("label")["video_id"].count().max()) if len(df) else 0,
    }

print("Split functions ready.")


## 7. Preprocessing Pose

Interpolasi dan normalisasi mengikuti flag eksperimen. Standardisasi temporal ke 64 frame selalu diterapkan karena merupakan kebutuhan format masukan model, bukan faktor eksperimen.


In [ ]:

# ============================================================
# 7. FUNGSI PREPROCESSING POSE
# ============================================================


def load_raw_pose_npz(raw_path: Path):
    """
    Load pose mentah. Referensi utama memakai key 'X_raw'.
    TODO: Jika output ekstraksi pose memakai nama key lain, tambahkan di daftar candidate_keys.
    """
    d = np.load(raw_path, allow_pickle=True)
    candidate_keys = ["X_raw", "X", "pose", "keypoints"]
    for k in candidate_keys:
        if k in d:
            X = d[k].astype(np.float32)
            break
    else:
        raise KeyError(f"Tidak menemukan key pose di {raw_path}. Key tersedia: {list(d.keys())}")

    # Ekspektasi utama: (T,65,2). Jika (2,T,65), ubah ke (T,65,2).
    if X.ndim != 3:
        raise ValueError(f"Shape pose tidak valid untuk {raw_path}: {X.shape}")
    if X.shape[-1] == 2:
        pass
    elif X.shape[0] == 2 and X.shape[-1] == N_JOINTS:
        X = np.transpose(X, (1, 2, 0))
    else:
        raise ValueError(f"Shape pose tidak dikenali untuk {raw_path}: {X.shape}")

    if X.shape[1] != N_JOINTS or X.shape[2] != N_COORDS:
        raise ValueError(f"Expected (T,{N_JOINTS},{N_COORDS}), got {X.shape} at {raw_path}")
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def detect_missing_mask(X):
    """True berarti joint missing. Missing didefinisikan sebagai koordinat (0,0)."""
    return (np.abs(X).sum(axis=-1) == 0.0)


def interpolate_missing_keypoints(X):
    """
    Interpolasi linear per joint dan per koordinat untuk missing keypoint.

    Keputusan final yang dipakai di notebook ini bersifat konservatif:
    - missing di tengah sequence diisi dengan interpolasi linear;
    - missing sebelum frame valid pertama tetap 0;
    - missing setelah frame valid terakhir tetap 0;
    - joint yang hilang sepanjang sequence atau hanya punya satu frame valid tetap 0 pada frame missing.

    Dengan demikian, interp=True tidak memaksa pose awal/akhir yang memang tidak terdeteksi.
    """
    X = X.copy().astype(np.float32)
    T, J, C = X.shape
    missing = detect_missing_mask(X)

    for j in range(J):
        valid = ~missing[:, j]
        if valid.all() or valid.sum() < 2:
            continue

        t_valid = np.where(valid)[0]
        t_fill = np.arange(t_valid[0], t_valid[-1] + 1)

        for c in range(C):
            y = X[t_valid, j, c]
            X[t_fill, j, c] = np.interp(t_fill, t_valid, y)

    return X.astype(np.float32)

def normalize_spatial(X, valid_mask=None):
    """
    Normalisasi geometrik:
    1. Geser semua joint relatif ke center.
    2. Scale dengan rata-rata jarak bahu.
    3. Restore joint missing menjadi 0 agar tidak menjadi ghost joint.
    """
    X = X.copy().astype(np.float32)
    if valid_mask is None:
        valid_mask = ~detect_missing_mask(X)
    valid_mask = valid_mask.copy()

    center = X[:, IDX_MID_HIP, :].copy()
    center_missing = ~valid_mask[:, IDX_MID_HIP]

    if center_missing.any():
        hip_center = (X[:, IDX_L_HIP, :] + X[:, IDX_R_HIP, :]) / 2.0
        hip_valid = valid_mask[:, IDX_L_HIP] & valid_mask[:, IDX_R_HIP]
        use_hip = center_missing & hip_valid
        center[use_hip] = hip_center[use_hip]
        center_missing = center_missing & ~use_hip

    if center_missing.any():
        shoulder_center = (X[:, IDX_L_SHOULDER, :] + X[:, IDX_R_SHOULDER, :]) / 2.0
        shoulder_valid = valid_mask[:, IDX_L_SHOULDER] & valid_mask[:, IDX_R_SHOULDER]
        use_shoulder = center_missing & shoulder_valid
        center[use_shoulder] = shoulder_center[use_shoulder]
        center_missing = center_missing & ~use_shoulder

    # Fallback terakhir: centroid joint valid per frame.
    if center_missing.any():
        for t in np.where(center_missing)[0]:
            pts = X[t, valid_mask[t]]
            center[t] = pts.mean(axis=0) if len(pts) else np.array([0.0, 0.0], dtype=np.float32)

    X = X - center[:, np.newaxis, :]

    l_sh = X[:, IDX_L_SHOULDER, :]
    r_sh = X[:, IDX_R_SHOULDER, :]
    shoulder_valid = valid_mask[:, IDX_L_SHOULDER] & valid_mask[:, IDX_R_SHOULDER]
    dists = np.linalg.norm(l_sh - r_sh, axis=-1)
    valid_dists = dists[shoulder_valid & np.isfinite(dists) & (dists > 1e-6)]
    mean_dist = valid_dists.mean() if len(valid_dists) > 0 else 1.0
    X = X / (mean_dist + 1e-6)
    X[~valid_mask] = 0.0
    return X.astype(np.float32)


def standardize_temporal_length(X, valid_mask, target_T=TARGET_T):
    """
    Standardisasi panjang sequence ke target_T.
    Koordinat di-resample linear, sementara mask valid di-resample dengan nearest neighbor.
    Setelah resize, joint yang mask-nya missing dikembalikan ke 0.
    """
    X = X.astype(np.float32)
    T = X.shape[0]
    valid_mask = valid_mask.astype(bool)
    if T == target_T:
        out = X.copy()
        out_mask = valid_mask.copy()
    else:
        t_old = np.linspace(0.0, 1.0, T)
        t_new = np.linspace(0.0, 1.0, target_T)
        out = np.zeros((target_T, N_JOINTS, N_COORDS), dtype=np.float32)
        for j in range(N_JOINTS):
            for c in range(N_COORDS):
                out[:, j, c] = interp1d(t_old, X[:, j, c], kind="linear", bounds_error=False, fill_value="extrapolate")(t_new)

        nearest_idx = np.clip(np.round(np.linspace(0, T - 1, target_T)).astype(int), 0, T - 1)
        out_mask = valid_mask[nearest_idx]

    out[~out_mask] = 0.0
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    return out.astype(np.float32), out_mask.astype(bool)


def preprocess_pose_base(X_raw, interp=False, norm=False):
    """
    Menghasilkan representasi pose sesuai konfigurasi eksperimen.
    - interp mengontrol interpolasi missing keypoint.
    - norm mengontrol normalisasi geometrik/spasial.
    - standardize_temporal_length() selalu aktif karena kebutuhan input model, bukan faktor eksperimen.
    """
    X = np.nan_to_num(X_raw.copy().astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    if interp:
        X = interpolate_missing_keypoints(X)

    # Mask valid dibuat setelah opsi interpolasi.
    # Dengan ini, joint yang berhasil diinterpolasi dianggap valid, sedangkan joint all-missing tetap missing.
    valid_mask = ~detect_missing_mask(X)

    if norm:
        X = normalize_spatial(X, valid_mask=valid_mask)

    X, valid_mask = standardize_temporal_length(X, valid_mask, target_T=TARGET_T)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[~valid_mask] = 0.0
    return X.astype(np.float32), valid_mask.astype(bool)


def to_model_shape(X_TVC):
    return np.transpose(X_TVC, (2, 0, 1)).astype(np.float32)  # (C,T,V)

print("Preprocessing functions ready.")


## 8. Build Artifact per Eksperimen dan Fold

Artifact disimpan per seed, eksperimen, dan fold. Completion flag serta pemeriksaan konfigurasi memungkinkan preprocessing dilanjutkan setelah runtime terputus tanpa mencampur artifact dari konfigurasi lain.


In [ ]:
# ============================================================
# 8. FUNGSI BUILD ARTEFAK PER EKSPERIMEN DAN FOLD
# ============================================================


def exp_data_dir(exp_id):
    return EXPERIMENT_DATA_ROOT / exp_id


def fold_data_dir(exp_id, fold):
    return exp_data_dir(exp_id) / f"fold{fold}"


def exp_run_dir(exp_id):
    return RUN_ROOT / exp_id


def fold_run_dir(exp_id, fold):
    return exp_run_dir(exp_id) / f"fold{fold}"


def preprocess_done_path(exp_id, fold):
    return fold_data_dir(exp_id, fold) / "PREPROCESS_DONE.txt"


def training_done_path(exp_id, fold):
    return fold_run_dir(exp_id, fold) / "DONE.txt"


def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def read_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)


def preprocess_config_matches(exp_id, fold):
    """Memastikan artefak preprocessing sesuai dengan registry eksperimen aktif."""
    cfg_path = fold_data_dir(exp_id, fold) / "preprocess_config.json"
    if not cfg_path.exists():
        return False
    try:
        saved = read_json(cfg_path)
    except Exception:
        return False

    cfg = EXPERIMENTS[exp_id]
    checks = {
        "interp": bool(cfg["interp"]),
        "norm": bool(cfg["norm"]),
        "augment": bool(cfg["augment"]),
        "target_T": int(TARGET_T),
        "seed": int(SEED),
    }

    for key, expected in checks.items():
        actual = saved.get(key)
        if actual != expected:
            return False
    return True


def build_eval_pack(df_subset, processed_by_vid, pack_path):
    Xs, ys, vids = [], [], []
    for _, row in df_subset.iterrows():
        vid = row["video_id"]
        if vid not in processed_by_vid:
            continue
        X_TVC, valid_mask = processed_by_vid[vid]
        Xs.append(to_model_shape(X_TVC))
        ys.append(int(row["label"]))
        vids.append(vid)
    if not Xs:
        raise RuntimeError(f"Pack kosong: {pack_path}")
    X = np.stack(Xs).astype(np.float32)
    y = np.array(ys, dtype=np.int64)
    np.savez_compressed(pack_path, X=X, y=y, video_id=np.array(vids, dtype=object))
    return X, y


def build_train_pack(df_subset, processed_by_vid, pack_path):
    """Pack train_fit agar training tidak perlu membaca banyak file kecil dari Google Drive."""
    Xs, ys, vids, masks = [], [], [], []
    for _, row in df_subset.iterrows():
        vid = row["video_id"]
        if vid not in processed_by_vid:
            continue
        X_TVC, valid_mask = processed_by_vid[vid]
        Xs.append(to_model_shape(X_TVC))
        ys.append(int(row["label"]))
        vids.append(vid)
        masks.append(valid_mask.astype(np.bool_))
    if not Xs:
        raise RuntimeError(f"Pack kosong: {pack_path}")
    X = np.stack(Xs).astype(np.float32)
    y = np.array(ys, dtype=np.int64)
    valid_mask = np.stack(masks).astype(np.bool_)
    np.savez_compressed(pack_path, X=X, y=y, video_id=np.array(vids, dtype=object), valid_mask=valid_mask)
    return X, y, valid_mask



def preprocess_fold_complete(exp_id, fold):
    """Preprocessing selesai hanya jika artefak utama ada dan config sesuai registry aktif."""
    fdir = fold_data_dir(exp_id, fold)
    required = [
        fdir / "PREPROCESS_DONE.txt",
        fdir / "train_index.csv",
        fdir / "train_fit_C2T64V65.npz",
        fdir / "val_internal_C2T64V65.npz",
        fdir / "test_eval_C2T64V65.npz",
        fdir / "preprocess_config.json",
    ]

    files_exist = all(p.exists() for p in required)
    return files_exist and preprocess_config_matches(exp_id, fold)


def build_artifacts_for_fold(exp_id, fold, df_meta, force=False):
    cfg = EXPERIMENTS[exp_id]
    fdir = fold_data_dir(exp_id, fold)
    train_samples_dir = fdir / "train_samples"
    fdir.mkdir(parents=True, exist_ok=True)
    train_samples_dir.mkdir(parents=True, exist_ok=True)

    done = preprocess_done_path(exp_id, fold)
    if preprocess_fold_complete(exp_id, fold) and not force:
        print(f"[SKIP PREPROCESS] {exp_id} fold{fold}: PREPROCESS_DONE dan artefak sesuai config aktif")
        return

    if done.exists() and not preprocess_fold_complete(exp_id, fold) and not force:
        print(f"[REBUILD PREPROCESS] {exp_id} fold{fold}: artefak ada, tetapi config tidak cocok dengan registry aktif. Fold akan dibangun ulang.")

    if force and fdir.exists():
        # Hapus output fold agar tidak mencampur artefak lama dan baru.
        shutil.rmtree(fdir)
        fdir.mkdir(parents=True, exist_ok=True)
        train_samples_dir = fdir / "train_samples"
        train_samples_dir.mkdir(parents=True, exist_ok=True)

    test_signer = fold
    df_train_fit, df_val_int, df_test = make_fold_split(df_meta, fold)
    assert_no_leakage(df_train_fit, df_val_int, df_test, test_signer)

    # Preprocess semua video yang dibutuhkan fold ini.
    df_needed = pd.concat([df_train_fit, df_val_int, df_test], axis=0).drop_duplicates("video_id")
    processed_final = {}

    for _, row in tqdm(df_needed.iterrows(), total=len(df_needed), desc=f"{exp_id} fold{fold} preprocess", leave=False):
        vid = row["video_id"]
        raw_path = VM_RAW_DIR / f"{vid}.npz"
        if not raw_path.exists():
            raise FileNotFoundError(f"Raw pose tidak ditemukan: {raw_path}")
        X_raw = load_raw_pose_npz(raw_path)
        X_final, valid_mask = preprocess_pose_base(X_raw, interp=cfg["interp"], norm=cfg["norm"])
        X_final = X_final.astype(np.float32)
        X_final[~valid_mask] = 0.0
        processed_final[vid] = (X_final, valid_mask)


    # Train index dan file per sample untuk Dataset training.
    train_rows = []
    for _, row in df_train_fit.iterrows():
        vid = row["video_id"]
        X_final, valid_mask = processed_final[vid]
        sample_path = train_samples_dir / f"{vid}.npz"
        np.savez_compressed(sample_path, X=to_model_shape(X_final), valid_mask=valid_mask.astype(np.bool_))
        train_rows.append({
            "video_id": vid,
            "signer_id": int(row["signer_id"]),
            "label": int(row["label"]),
            "sample_id": int(row["sample_id"]),
            "sample_path": str(sample_path),
            "data_key": "X",
        })

    df_train_index = pd.DataFrame(train_rows)
    df_train_index.to_csv(fdir / "train_index.csv", index=False)

    # Simpan metadata audit split.
    df_train_fit.to_csv(fdir / "train_fit_meta.csv", index=False)
    df_val_int.to_csv(fdir / "val_internal_meta.csv", index=False)
    df_test.to_csv(fdir / "test_meta.csv", index=False)

    train_X, train_y, train_valid_mask = build_train_pack(df_train_fit, processed_final, fdir / "train_fit_C2T64V65.npz")
    val_X, val_y = build_eval_pack(df_val_int, processed_final, fdir / "val_internal_C2T64V65.npz")
    test_X, test_y = build_eval_pack(df_test, processed_final, fdir / "test_eval_C2T64V65.npz")

    split_audit = []
    for subset_name, subset_df in [("train_fit", df_train_fit), ("val_internal", df_val_int), ("test", df_test)]:
        srow = summarize_split(subset_df, subset_name)
        srow["fold"] = fold
        srow["test_signer"] = test_signer
        split_audit.append(srow)
    pd.DataFrame(split_audit).to_csv(fdir / "split_audit.csv", index=False)

    preprocess_config = {
        "exp_id": exp_id,
        "name": cfg["name"],
        "fold": fold,
        "test_signer": test_signer,
        "interp": bool(cfg["interp"]),
        "norm": bool(cfg["norm"]),
        "augment": bool(cfg["augment"]),
        "target_T": int(TARGET_T),
        "n_joints": int(N_JOINTS),
        "n_coords": int(N_COORDS),
        "val_frac": float(VAL_FRAC),
        "seed": int(SEED),
        "normalization": "geometric_spatial" if cfg["norm"] else None,
        "interpolation": "linear_missing_keypoint" if cfg["interp"] else None,
        "augment_application": "train_fit_dataset_only" if cfg["augment"] else None,
        "train_n": int(len(df_train_fit)),
        "val_internal_n": int(len(df_val_int)),
        "test_n": int(len(df_test)),
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "note": "Temporal resize ke 64 frame aktif di semua eksperimen. Normalisasi yang digunakan adalah normalisasi geometrik/spasial.",
    }
    write_json(fdir / "preprocess_config.json", preprocess_config)

    # DONE ditulis paling akhir agar resume aman.
    done.write_text(datetime.now().isoformat(timespec="seconds"))
    print(f"[PREPROCESS DONE] {exp_id} fold{fold} | train_pack={train_X.shape} | val={val_X.shape} | test={test_X.shape}")


def build_artifacts_for_experiment(exp_id, force=False):
    exp_cfg = {"exp_id": exp_id, "seed": int(SEED), **EXPERIMENTS[exp_id], "hyperparams": HYPERPARAMS, "augment_config": AUGMENT_CONFIG}
    exp_data_dir(exp_id).mkdir(parents=True, exist_ok=True)
    exp_run_dir(exp_id).mkdir(parents=True, exist_ok=True)
    write_json(exp_run_dir(exp_id) / "config.json", exp_cfg)
    for fold in TEST_SIGNERS:
        build_artifacts_for_fold(exp_id, fold, df_meta, force=force)

print("Artifact build functions ready.")


## 9. QC Missing Keypoint Artifact

Missing keypoint didefinisikan sebagai joint dengan koordinat `x=0` dan `y=0`. QC ini digunakan untuk melihat proporsi missing keypoint pada artifact hasil preprocessing.

Pada eksperimen dengan `interp=True`, missing keypoint yang berada di antara dua frame valid diisi menggunakan interpolasi linear temporal. Missing keypoint pada awal/akhir sequence atau joint yang tidak memiliki informasi valid yang cukup tetap dipertahankan sebagai `0`.

QC ini tidak digunakan untuk mengubah data, model, split, hyperparameter, training loop, atau hasil eksperimen. QC hanya membaca artifact `.npz` dan menyimpan laporan audit.

In [ ]:

# ============================================================
# 9. QC MISSING KEYPOINT ARTIFACT
# ============================================================


def compute_missing_rate(X):
    """
    Menghitung missing keypoint rate.

    Missing keypoint didefinisikan sebagai joint dengan x=0 dan y=0.

    Mendukung shape:
    - (N, C, T, V)
    - (C, T, V)

    Dengan C=2, yaitu koordinat x dan y.
    """
    X = np.asarray(X)

    if X.ndim == 4:
        missing_mask = (X[:, 0] == 0) & (X[:, 1] == 0)
    elif X.ndim == 3:
        missing_mask = (X[0] == 0) & (X[1] == 0)
    else:
        raise ValueError(f"Unsupported X shape: {X.shape}")

    return float(missing_mask.mean())


def audit_missing_rate_pack(npz_path):
    """
    Menghitung missing keypoint rate dari file pack .npz.
    File diharapkan memiliki key 'X'.
    """
    npz_path = Path(npz_path)

    if not npz_path.exists():
        return {
            "path": str(npz_path),
            "exists": False,
            "missing_rate": np.nan,
            "n_samples": 0,
        }

    pack = np.load(npz_path, allow_pickle=True)
    X = pack["X"]

    return {
        "path": str(npz_path),
        "exists": True,
        "missing_rate": compute_missing_rate(X),
        "n_samples": int(X.shape[0]) if X.ndim == 4 else 1,
        "shape": tuple(X.shape),
    }


def audit_missing_rate_experiment(exp_id):
    """
    Membuat ringkasan missing keypoint rate untuk semua fold pada satu eksperimen.
    Audit dilakukan pada train_fit, val_internal, dan test_eval.
    """
    rows = []

    for fold in TEST_SIGNERS:
        fdir = fold_data_dir(exp_id, fold)

        packs = {
            "train_fit": fdir / "train_fit_C2T64V65.npz",
            "val_internal": fdir / "val_internal_C2T64V65.npz",
            "test_eval": fdir / "test_eval_C2T64V65.npz",
        }

        for split_name, path in packs.items():
            result = audit_missing_rate_pack(path)
            rows.append({
                "exp_id": exp_id,
                "fold": fold,
                "split": split_name,
                "exists": result["exists"],
                "n_samples": result.get("n_samples", 0),
                "shape": result.get("shape", None),
                "missing_rate": result["missing_rate"],
                "missing_rate_pct": result["missing_rate"] * 100 if result["exists"] else np.nan,
                "path": result["path"],
            })

    df = pd.DataFrame(rows)
    return df


def save_missing_rate_report(exp_id):
    """
    Menyimpan hasil QC missing keypoint untuk satu eksperimen.
    """
    df = audit_missing_rate_experiment(exp_id)

    out_path = exp_data_dir(exp_id) / f"missing_rate_report_{exp_id}.csv"
    df.to_csv(out_path, index=False)

    print(f"[QC] Missing keypoint report saved: {out_path}")
    display(df)

    return df


def compare_missing_rate_between_experiments(exp_ids):
    """
    Membandingkan missing keypoint rate beberapa eksperimen.
    Berguna untuk membandingkan E0 vs E1, E0 vs E4, atau eksperimen interp=False vs interp=True.
    """
    all_rows = []

    for exp_id in exp_ids:
        df = audit_missing_rate_experiment(exp_id)
        all_rows.append(df)

    df_all = pd.concat(all_rows, ignore_index=True)

    summary = (
        df_all
        .groupby(["exp_id", "split"], as_index=False)
        .agg(
            mean_missing_rate=("missing_rate", "mean"),
            std_missing_rate=("missing_rate", "std"),
            min_missing_rate=("missing_rate", "min"),
            max_missing_rate=("missing_rate", "max"),
        )
    )

    summary["mean_missing_rate_pct"] = summary["mean_missing_rate"] * 100
    summary["std_missing_rate_pct"] = summary["std_missing_rate"] * 100

    out_path = EXPERIMENT_DATA_ROOT / "missing_rate_comparison.csv"
    summary.to_csv(out_path, index=False)

    print(f"[QC] Missing keypoint comparison saved: {out_path}")
    display(summary)

    return summary


print("QC missing keypoint functions ready.")


## 10. QC Visual Artifact

QC visual digunakan untuk memeriksa skeleton hasil preprocessing secara kualitatif dan frame-by-frame. QC ini membantu memastikan bahwa interpolasi *missing keypoint*, normalisasi, dan *temporal standardization* berjalan sesuai harapan.

QC visual tidak digunakan sebagai metrik evaluasi model. Fungsi pada section ini hanya membaca file artifact `.npz`, tidak menyimpan perubahan ke data, dan tidak mengubah model maupun training loop.

Mode QC visual utama pada notebook ini adalah **same-label all-signers QC**:

- memilih satu label/gloss yang sama;
- mengambil satu sample dari setiap signer `0, 1, 2, 3, 4` jika tersedia;
- menampilkan beberapa frame representatif dari setiap sample;
- membandingkan `raw_source_T64`, `artifact`, dan `augmented_preview` jika eksperimen memakai augmentasi.

Karena signer 0 hanya memiliki sebagian label, label QC sebaiknya dipilih dari label yang tersedia pada semua signer. Berdasarkan kondisi dataset saat ini, pilihan label yang aman berada pada rentang 0–11.


Catatan revisi QC visual: setiap gambar QC sekarang memakai *fixed axis* agar Matplotlib tidak melakukan *auto-zoom* pada setiap subplot. Dengan demikian, variasi posisi dan skala pose pada `raw_source_T64` atau artifact `norm=False` tetap terlihat. Mode default adalah `stage_fixed`, yaitu axis dibuat sama untuk stage yang sama (`raw_source_T64`, `artifact`, atau `augmented_preview`) pada seluruh signer dan frame yang diplot.

In [ ]:

# ============================================================
# 10. QC VISUAL ARTIFACT
# ============================================================


def get_pack_path(exp_id, fold, split="train_fit"):
    """Mengambil path pack artifact berdasarkan exp_id, fold, dan split."""
    fdir = fold_data_dir(exp_id, fold)

    if split == "train_fit":
        return fdir / "train_fit_C2T64V65.npz"
    if split == "val_internal":
        return fdir / "val_internal_C2T64V65.npz"
    if split == "test_eval":
        return fdir / "test_eval_C2T64V65.npz"

    raise ValueError(f"Unknown split: {split}")


def load_sample_from_pack(exp_id, fold, split="train_fit", sample_idx=0):
    path = get_pack_path(exp_id, fold, split)
    pack = np.load(path, allow_pickle=True)
    X = pack["X"]
    y = pack["y"]

    sample = X[sample_idx]
    label = int(y[sample_idx])

    video_id = None
    for key in ["video_id", "video_ids", "ids"]:
        if key in pack:
            video_id = str(pack[key][sample_idx])
            break

    signer_id = None
    sample_id = None
    if video_id is not None:
        parsed = parse_video_id(video_id)
        if parsed is not None:
            signer_id, parsed_label, sample_id = parsed
            if parsed_label != label:
                print(f"[WARN] Label pack ({label}) berbeda dengan label video_id ({parsed_label}) untuk {video_id}")

    return sample, label, video_id, path


def compute_pose_axis_limits(X_list, frames=None, padding_ratio=0.08, force_square=True):
    """
    Menghitung batas axis tetap untuk QC visual.

    Missing keypoint (x=0 dan y=0) diabaikan dari perhitungan axis agar
    titik placeholder tidak membuat tampilan bias ke koordinat 0.
    """
    xs = []
    ys = []

    for X in X_list:
        if X is None:
            continue
        X = np.asarray(X)
        if X.ndim != 3:
            raise ValueError(f"Expected X shape (C,T,V), got {X.shape}")

        use_frames = range(X.shape[1]) if frames is None else frames
        for frame_idx in use_frames:
            frame_idx = int(frame_idx)
            if frame_idx < 0 or frame_idx >= X.shape[1]:
                continue
            x = X[0, frame_idx]
            y = X[1, frame_idx]
            valid = (x != 0) | (y != 0)
            if np.any(valid):
                xs.append(x[valid])
                ys.append(y[valid])

    if not xs or not ys:
        return None

    x_all = np.concatenate(xs).astype(float)
    y_all = np.concatenate(ys).astype(float)
    x_all = x_all[np.isfinite(x_all)]
    y_all = y_all[np.isfinite(y_all)]

    if len(x_all) == 0 or len(y_all) == 0:
        return None

    xmin, xmax = float(np.min(x_all)), float(np.max(x_all))
    ymin, ymax = float(np.min(y_all)), float(np.max(y_all))

    x_span = max(xmax - xmin, 1e-6)
    y_span = max(ymax - ymin, 1e-6)

    if force_square:
        span = max(x_span, y_span)
        x_center = (xmin + xmax) / 2.0
        y_center = (ymin + ymax) / 2.0
        pad = span * float(padding_ratio)
        half = span / 2.0 + pad
        xmin, xmax = x_center - half, x_center + half
        ymin, ymax = y_center - half, y_center + half
    else:
        xmin -= x_span * float(padding_ratio)
        xmax += x_span * float(padding_ratio)
        ymin -= y_span * float(padding_ratio)
        ymax += y_span * float(padding_ratio)

    return {
        "xlim": (xmin, xmax),
        "ylim": (ymin, ymax),
        "x_min": xmin,
        "x_max": xmax,
        "y_min": ymin,
        "y_max": ymax,
    }


def axis_limits_to_row(stage, limits, axis_mode):
    """Mengubah batas axis menjadi row manifest."""
    if limits is None:
        return {
            "stage": stage,
            "axis_mode": axis_mode,
            "x_min": np.nan,
            "x_max": np.nan,
            "y_min": np.nan,
            "y_max": np.nan,
        }
    return {
        "stage": stage,
        "axis_mode": axis_mode,
        "x_min": limits["x_min"],
        "x_max": limits["x_max"],
        "y_min": limits["y_min"],
        "y_max": limits["y_max"],
    }


def resolve_axis_limits_for_rows(rows_for_plot, frames, axis_mode="stage_fixed"):
    """
    Menentukan axis tetap untuk rows QC visual.

    axis_mode:
    - "stage_fixed": axis sama untuk stage yang sama di seluruh signer/frame.
    - "global_fixed": axis sama untuk seluruh subplot dalam satu gambar.
    - "auto": axis otomatis Matplotlib, tidak direkomendasikan untuk cek posisi/skala.
    """
    axis_mode = str(axis_mode)
    if axis_mode not in ["stage_fixed", "global_fixed", "auto"]:
        raise ValueError("axis_mode harus salah satu dari: stage_fixed, global_fixed, auto")

    if axis_mode == "auto":
        return {}, []

    if axis_mode == "global_fixed":
        limits = compute_pose_axis_limits([row[2] for row in rows_for_plot], frames=frames)
        axis_rows = [axis_limits_to_row("__global__", limits, axis_mode)]
        return {"__global__": limits}, axis_rows

    limits_by_stage = {}
    axis_rows = []
    stages = sorted(set(stage for _, stage, _ in rows_for_plot))
    for stage in stages:
        X_list = [X for _, row_stage, X in rows_for_plot if row_stage == stage]
        limits = compute_pose_axis_limits(X_list, frames=frames)
        limits_by_stage[stage] = limits
        axis_rows.append(axis_limits_to_row(stage, limits, axis_mode))

    return limits_by_stage, axis_rows


def get_limits_for_stage(axis_limits, stage, axis_mode):
    """Mengambil xlim/ylim untuk stage tertentu."""
    if axis_mode == "auto":
        return None
    if axis_mode == "global_fixed":
        return axis_limits.get("__global__")
    return axis_limits.get(stage)


def plot_pose_frame(
    X,
    frame_idx=0,
    title=None,
    edges=None,
    ax=None,
    xlim=None,
    ylim=None,
    show_coord_range=False,
):
    """
    X shape: (C, T, V), C=2.
    Missing keypoint didefinisikan sebagai x=0 dan y=0.

    Parameter xlim/ylim dipakai agar QC visual tidak auto-zoom per subplot.
    """
    import matplotlib.pyplot as plt

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    x = X[0, frame_idx]
    y = X[1, frame_idx]

    valid = (x != 0) | (y != 0)

    ax.scatter(x[valid], y[valid], s=12)

    if edges is not None:
        for i, j in edges:
            if valid[i] and valid[j]:
                ax.plot([x[i], x[j]], [y[i], y[j]], linewidth=1)

    ax.set_title(title or f"Frame {frame_idx}")
    ax.set_aspect("equal", adjustable="box")

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

    if show_coord_range and np.any(valid):
        note = (
            f"x:[{float(np.min(x[valid])):.3f},{float(np.max(x[valid])):.3f}]; "
            f"y:[{float(np.min(y[valid])):.3f},{float(np.max(y[valid])):.3f}]"
        )
        ax.text(
            0.02,
            0.98,
            note,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=7,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.65},
        )

    return ax


def visual_qc_sample(exp_id, fold=1, split="train_fit", sample_idx=0, frames=None, edges=None):
    """
    Menampilkan beberapa frame dari satu sample hasil preprocessing.
    Default menampilkan 5 frame representatif, bukan semua 64 frame.
    """
    import matplotlib.pyplot as plt

    X, label, video_id, path = load_sample_from_pack(exp_id, fold, split, sample_idx)

    if frames is None:
        T = X.shape[1]
        frames = [0, T // 4, T // 2, 3 * T // 4, T - 1]

    axis_info = compute_pose_axis_limits([X], frames=frames)
    xlim = None if axis_info is None else axis_info["xlim"]
    ylim = None if axis_info is None else axis_info["ylim"]

    n = len(frames)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))

    if n == 1:
        axes = [axes]

    for ax, frame_idx in zip(axes, frames):
        plot_pose_frame(
            X,
            frame_idx=frame_idx,
            title=f"{exp_id} | {split} | f={frame_idx}",
            edges=edges,
            ax=ax,
            xlim=xlim,
            ylim=ylim,
            show_coord_range=True,
        )

    fig.suptitle(
        f"exp={exp_id}, fold={fold}, split={split}, idx={sample_idx}, "
        f"label={label}, video_id={video_id}"
    )
    plt.tight_layout()
    plt.show()

    return X


def visual_compare_experiments(exp_ids, fold=1, split="train_fit", sample_idx=0, frame_idx=0, edges=None, axis_mode="global_fixed"):
    """
    Membandingkan frame yang sama dari beberapa eksperimen.
    Cocok untuk E0 vs E1 atau E0 vs E4.

    Default axis_mode="global_fixed" agar skala/posisi antar eksperimen tidak terkena auto-zoom.
    """
    import matplotlib.pyplot as plt

    loaded = []
    for exp_id in exp_ids:
        X, label, video_id, path = load_sample_from_pack(exp_id, fold, split, sample_idx)
        loaded.append((exp_id, X, label, video_id, path))

    rows_for_plot = [({}, str(exp_id), X) for exp_id, X, _, _, _ in loaded]
    axis_limits, _ = resolve_axis_limits_for_rows(rows_for_plot, frames=[frame_idx], axis_mode=axis_mode)

    n = len(exp_ids)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    if n == 1:
        axes = [axes]

    for ax, (exp_id, X, label, video_id, path) in zip(axes, loaded):
        limits = get_limits_for_stage(axis_limits, str(exp_id), axis_mode)
        plot_pose_frame(
            X,
            frame_idx=frame_idx,
            title=f"{exp_id} | frame={frame_idx}",
            edges=edges,
            ax=ax,
            xlim=None if limits is None else limits["xlim"],
            ylim=None if limits is None else limits["ylim"],
            show_coord_range=True,
        )

    fig.suptitle(
        f"Compare exp={exp_ids}, fold={fold}, split={split}, "
        f"idx={sample_idx}, frame={frame_idx}, axis_mode={axis_mode}"
    )
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Fungsi simpan QC visual ke Drive
# ------------------------------------------------------------


def safe_filename_text(x):
    """Membuat teks aman untuk nama file."""
    if x is None:
        return "none"
    text = str(x)
    keep = []
    for ch in text:
        if ch.isalnum() or ch in ["-", "_", "."]:
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)[:120]


def qc_visual_dir(exp_id):
    """Folder QC visual untuk satu eksperimen."""
    return exp_data_dir(exp_id) / "QC"


def resolve_qc_edges(edges=None):
    """Mengambil edge list jika sudah tersedia. Jika belum, return None."""
    if edges is not None:
        return edges
    if "EDGE_LIST" in globals():
        return globals()["EDGE_LIST"]
    if "EDGES" in globals():
        return globals()["EDGES"]
    return None


def load_raw_source_T64_from_video_id(video_id):
    """
    Membaca raw pose berdasarkan video_id lalu menstandardisasi panjang sequence ke TARGET_T.
    Fungsi ini hanya menerapkan nan/inf safety dan standardisasi panjang sequence untuk menampilkan sumber pose mentah.
    Output shape: (C, T, V).
    """
    if video_id is None:
        return None

    vid = str(video_id)
    candidates = [
        VM_RAW_DIR / f"{vid}.npz",
        VM_RAW_DIR / vid,
    ]
    if not vid.endswith(".npz"):
        candidates.append(VM_RAW_DIR / f"{safe_filename_text(vid)}.npz")

    raw_path = None
    for cand in candidates:
        if cand.exists():
            raw_path = cand
            break

    if raw_path is None:
        return None

    X_raw = load_raw_pose_npz(raw_path)
    valid_mask = ~detect_missing_mask(X_raw)
    X_T64, _ = standardize_temporal_length(X_raw, valid_mask, target_T=TARGET_T)
    return to_model_shape(X_T64)


def make_augmented_preview_for_qc(X, seed_offset=0):
    """
    Membuat preview augmentasi tanpa mengubah state random global.
    Preview hanya untuk dokumentasi visual, bukan untuk training/evaluasi.
    """
    np_state = np.random.get_state()
    try:
        np.random.seed(SEED + int(seed_offset))
        valid_mask = valid_mask_ctv_from_coords(X)
        X_aug = apply_pose_spatial_aug_v1(X.copy(), valid_mask=valid_mask)
    finally:
        np.random.set_state(np_state)
    return X_aug.astype(np.float32)


def get_common_labels_for_signers(signers=None):
    """
    Mengambil label yang tersedia untuk seluruh signer yang diminta.
    Berguna karena signer 0 tidak memiliki seluruh 32 label.
    """
    signers = ALL_SIGNERS if signers is None else list(signers)
    label_sets = []
    for signer_id in signers:
        labs = set(df_meta.loc[df_meta["signer_id"] == signer_id, "label"].astype(int).tolist())
        label_sets.append(labs)

    if not label_sets:
        return []

    common = sorted(set.intersection(*label_sets))
    return common


def find_sample_index_in_pack_by_signer_label(exp_id, fold, split, signer_id, label):
    """
    Mencari index sample dalam satu pack berdasarkan signer_id dan label.
    signer_id dibaca dari video_id menggunakan parse_video_id().
    """
    pack_path = get_pack_path(exp_id, fold, split)
    if not pack_path.exists():
        return None

    pack = np.load(pack_path, allow_pickle=True)
    y = pack["y"].astype(int)

    if "video_id" in pack:
        video_ids = pack["video_id"].astype(str)
    elif "video_ids" in pack:
        video_ids = pack["video_ids"].astype(str)
    elif "ids" in pack:
        video_ids = pack["ids"].astype(str)
    else:
        print(f"[WARN] Pack tidak memiliki video_id: {pack_path}")
        return None

    for idx, (vid, lab) in enumerate(zip(video_ids, y)):
        parsed = parse_video_id(vid)
        if parsed is None:
            continue
        s, parsed_label, sample_id = parsed
        if int(s) == int(signer_id) and int(lab) == int(label):
            return {
                "exp_id": exp_id,
                "fold": int(fold),
                "split": split,
                "sample_idx": int(idx),
                "signer_id": int(signer_id),
                "label": int(label),
                "video_id": str(vid),
                "sample_id": int(sample_id),
                "pack_path": str(pack_path),
            }

    return None


def find_qc_sample_for_signer_label(exp_id, signer_id, label, preferred_split="train_fit"):
    """
    Mencari satu sample untuk signer dan label tertentu.

    Prioritas utama adalah train_fit agar preview augmentasi dapat dibuat untuk semua signer.
    Karena LOSO menahan satu signer sebagai test pada fold tertentu, fungsi ini memilih fold lain
    ketika signer tersebut menjadi test signer.
    """
    fold_priority = list(TEST_SIGNERS)

    split_priority = []
    if preferred_split:
        split_priority.append(preferred_split)
    for split in ["train_fit", "val_internal", "test_eval"]:
        if split not in split_priority:
            split_priority.append(split)

    for split in split_priority:
        for fold in fold_priority:
            # Pada train_fit dan val_internal, test signer tidak boleh muncul.
            if split in ["train_fit", "val_internal"] and int(signer_id) == int(fold):
                continue
            # Pada test_eval, hanya test signer yang muncul.
            if split == "test_eval" and int(signer_id) != int(fold):
                continue

            found = find_sample_index_in_pack_by_signer_label(
                exp_id=exp_id,
                fold=fold,
                split=split,
                signer_id=signer_id,
                label=label,
            )
            if found is not None:
                return found

    return None


def collect_same_label_all_signers_qc_samples(exp_id, label=0, signers=None, preferred_split="train_fit"):
    """
    Mengumpulkan satu sample untuk setiap signer pada label yang sama.
    """
    signers = ALL_SIGNERS if signers is None else list(signers)
    common_labels = get_common_labels_for_signers(signers)

    if int(label) not in common_labels:
        raise ValueError(
            f"Label {label} tidak tersedia untuk seluruh signer {signers}. "
            f"Pilihan label yang tersedia untuk semua signer: {common_labels}"
        )

    rows = []
    missing = []
    for signer_id in signers:
        found = find_qc_sample_for_signer_label(
            exp_id=exp_id,
            signer_id=signer_id,
            label=label,
            preferred_split=preferred_split,
        )
        if found is None:
            missing.append(signer_id)
        else:
            rows.append(found)

    if missing:
        raise RuntimeError(f"Tidak menemukan sample QC untuk signer {missing}, label={label}, exp={exp_id}")

    return rows


def save_qc_visual_same_label_all_signers(
    exp_id,
    label=0,
    signers=None,
    frames=None,
    edges=None,
    preferred_split="train_fit",
    include_raw=True,
    include_augmented_preview=True,
    axis_mode="stage_fixed",
    dpi=120,
    preview=True,
):
    """
    Menyimpan QC visual untuk satu label yang sama dan seluruh signer.

    Output berupa satu gambar besar berisi satu sample per signer.
    Setiap signer dapat memiliki baris:
    - raw_source_T64
    - artifact
    - augmented_preview, hanya jika eksperimen augment=True dan sample ditemukan pada train_fit.

    Default frame yang disimpan adalah 5 frame representatif, bukan seluruh 64 frame.
    """
    import matplotlib.pyplot as plt
    from IPython.display import Image as IPyImage

    edges = resolve_qc_edges(edges)
    signers = ALL_SIGNERS if signers is None else list(signers)
    if frames is None:
        frames = [0, 10, 20, 40, TARGET_T - 1]
    frames = [int(f) for f in frames if 0 <= int(f) < TARGET_T]

    selected = collect_same_label_all_signers_qc_samples(
        exp_id=exp_id,
        label=label,
        signers=signers,
        preferred_split=preferred_split,
    )

    rows_for_plot = []
    manifest_rows = []
    cfg = EXPERIMENTS[exp_id]

    for row in selected:
        X_art, y, video_id, pack_path = load_sample_from_pack(
            exp_id=exp_id,
            fold=row["fold"],
            split=row["split"],
            sample_idx=row["sample_idx"],
        )

        stage_count = 0
        if include_raw:
            X_raw_T64 = load_raw_source_T64_from_video_id(video_id)
            if X_raw_T64 is not None:
                rows_for_plot.append((row, "raw_source_T64", X_raw_T64))
                stage_count += 1
            else:
                print(f"[WARN] Raw source tidak ditemukan untuk video_id={video_id}")

        rows_for_plot.append((row, "artifact", X_art))
        stage_count += 1

        if include_augmented_preview and cfg.get("augment", False) and row["split"] == "train_fit":
            seed_offset = 100000 + row["fold"] * 1000 + int(row["sample_idx"])
            X_aug = make_augmented_preview_for_qc(X_art, seed_offset=seed_offset)
            rows_for_plot.append((row, "augmented_preview", X_aug))
            stage_count += 1

        manifest_row = dict(row)
        manifest_row["n_stages_plotted"] = stage_count
        manifest_rows.append(manifest_row)

    axis_limits, axis_rows = resolve_axis_limits_for_rows(
        rows_for_plot,
        frames=frames,
        axis_mode=axis_mode,
    )

    n_rows = len(rows_for_plot)
    n_cols = len(frames)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 3.2 * n_rows),
        squeeze=False,
    )

    for r, (meta, stage, X_row) in enumerate(rows_for_plot):
        for c, frame_idx in enumerate(frames):
            ax = axes[r][c]
            title = (
                f"signer={meta['signer_id']} | {stage} | f={frame_idx}\n"
                f"fold={meta['fold']} | {meta['split']} | sample={meta['sample_id']}"
            )
            limits = get_limits_for_stage(axis_limits, stage, axis_mode)
            plot_pose_frame(
                X_row,
                frame_idx=frame_idx,
                title=title,
                edges=edges,
                ax=ax,
                xlim=None if limits is None else limits["xlim"],
                ylim=None if limits is None else limits["ylim"],
                show_coord_range=True,
            )

    fig.suptitle(
        f"QC Visual Same Label All Signers | exp={exp_id} | label={label} | "
        f"frames={frames} | axis_mode={axis_mode}",
        fontsize=14,
    )
    plt.tight_layout()

    out_dir = qc_visual_dir(exp_id) / "same_label_all_signers"
    out_dir.mkdir(parents=True, exist_ok=True)
    frame_txt = "_".join(map(str, frames))
    signer_txt = "_".join(map(str, signers))
    out_path = out_dir / f"{exp_id}_label{label}_signers_{signer_txt}_frames_{frame_txt}_{axis_mode}.png"
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    df_manifest = pd.DataFrame(manifest_rows)
    manifest_path = out_dir / f"{exp_id}_label{label}_same_label_all_signers_manifest.csv"
    df_manifest.to_csv(manifest_path, index=False)

    df_axis = pd.DataFrame(axis_rows)
    axis_report_path = out_dir / f"{exp_id}_label{label}_axis_report_{axis_mode}.csv"
    df_axis.to_csv(axis_report_path, index=False)

    print(f"[QC VISUAL] Same-label all-signers image saved: {out_path}")
    print(f"[QC VISUAL] Manifest saved: {manifest_path}")
    print(f"[QC VISUAL] Axis report saved: {axis_report_path}")

    if preview:
        print("\n[QC VISUAL PREVIEW]")
        display(IPyImage(filename=str(out_path)))

    display(df_manifest)
    display(df_axis)

    return df_manifest


def save_qc_visual_artifact(
    exp_id,
    fold=1,
    split="train_fit",
    sample_idx=0,
    frames=None,
    edges=None,
    include_raw=True,
    include_augmented_preview=True,
    axis_mode="global_fixed",
    dpi=120,
):
    """
    Menyimpan QC visual untuk satu sample tertentu.
    Fungsi ini dipertahankan untuk kebutuhan inspeksi manual.
    """
    import matplotlib.pyplot as plt

    edges = resolve_qc_edges(edges)
    X_art, label, video_id, pack_path = load_sample_from_pack(exp_id, fold, split, sample_idx)

    if frames is None:
        T = X_art.shape[1]
        frames = [0, T // 4, T // 2, 3 * T // 4, T - 1]
    frames = [int(f) for f in frames if 0 <= int(f) < X_art.shape[1]]

    rows = []
    if include_raw:
        X_raw_T64 = load_raw_source_T64_from_video_id(video_id)
        if X_raw_T64 is not None:
            rows.append(("raw_source_T64", X_raw_T64))
        else:
            print(f"[WARN] Raw source tidak ditemukan untuk video_id={video_id} ({exp_id} fold{fold} {split} idx{sample_idx})")

    rows.append(("artifact", X_art))

    cfg = EXPERIMENTS[exp_id]
    if include_augmented_preview and cfg.get("augment", False) and split == "train_fit":
        seed_offset = 100000 + fold * 1000 + int(sample_idx)
        X_aug = make_augmented_preview_for_qc(X_art, seed_offset=seed_offset)
        rows.append(("augmented_preview", X_aug))

    rows_for_axis = [({}, row_name, X_row) for row_name, X_row in rows]
    axis_limits, axis_rows = resolve_axis_limits_for_rows(rows_for_axis, frames=frames, axis_mode=axis_mode)

    n_rows = len(rows)
    n_cols = len(frames)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), squeeze=False)

    for r, (row_name, X_row) in enumerate(rows):
        for c, frame_idx in enumerate(frames):
            ax = axes[r][c]
            limits = get_limits_for_stage(axis_limits, row_name, axis_mode)
            plot_pose_frame(
                X_row,
                frame_idx=frame_idx,
                title=f"{row_name} | f={frame_idx}",
                edges=edges,
                ax=ax,
                xlim=None if limits is None else limits["xlim"],
                ylim=None if limits is None else limits["ylim"],
                show_coord_range=True,
            )

    fig.suptitle(
        f"QC Visual | exp={exp_id} | fold={fold} | split={split} | "
        f"idx={sample_idx} | label={label} | video_id={video_id} | axis_mode={axis_mode}",
        fontsize=12,
    )
    plt.tight_layout()

    out_dir = qc_visual_dir(exp_id) / f"fold{fold}" / split
    out_dir.mkdir(parents=True, exist_ok=True)
    out_name = (
        f"{exp_id}_fold{fold}_{split}_idx{sample_idx}_"
        f"label{label}_video_{safe_filename_text(video_id)}_{axis_mode}.png"
    )
    out_path = out_dir / out_name
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    axis_report_path = out_dir / f"{exp_id}_fold{fold}_{split}_idx{sample_idx}_axis_report_{axis_mode}.csv"
    pd.DataFrame(axis_rows).to_csv(axis_report_path, index=False)

    return {
        "exp_id": exp_id,
        "fold": fold,
        "split": split,
        "sample_idx": sample_idx,
        "label": label,
        "video_id": video_id,
        "path": str(out_path),
        "axis_report_path": str(axis_report_path),
    }


def save_visual_qc_experiment(
    exp_id,
    folds=None,
    splits=("train_fit", "val_internal", "test_eval"),
    max_samples_per_split=3,
    frames=None,
    edges=None,
    include_raw=True,
    include_augmented_preview=True,
    axis_mode="global_fixed",
):
    """
    Menyimpan QC visual banyak sample untuk satu eksperimen.
    Fungsi ini dipertahankan sebagai opsi manual, tetapi bukan mode QC utama.
    """
    folds = TEST_SIGNERS if folds is None else folds
    rows = []

    for fold in folds:
        for split in splits:
            pack_path = get_pack_path(exp_id, fold, split)

            if not pack_path.exists():
                print(f"[QC VISUAL SKIP] Pack belum ada: {pack_path}")
                continue

            pack = np.load(pack_path, allow_pickle=True)
            n_samples = int(pack["X"].shape[0])
            limit = n_samples if max_samples_per_split is None else min(int(max_samples_per_split), n_samples)

            print(f"[QC VISUAL] {exp_id} fold{fold} {split}: menyimpan {limit}/{n_samples} sample")
            for sample_idx in range(limit):
                row = save_qc_visual_artifact(
                    exp_id=exp_id,
                    fold=fold,
                    split=split,
                    sample_idx=sample_idx,
                    frames=frames,
                    edges=edges,
                    include_raw=include_raw,
                    include_augmented_preview=include_augmented_preview,
                    axis_mode=axis_mode,
                )
                rows.append(row)

    df_manifest = pd.DataFrame(rows)
    out_path = qc_visual_dir(exp_id) / f"qc_visual_manifest_{exp_id}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_manifest.to_csv(out_path, index=False)
    print(f"[QC VISUAL] Manifest saved: {out_path}")
    if len(df_manifest):
        display(df_manifest.head())
    return df_manifest


# Contoh penggunaan manual setelah artifact selesai dibuat.
# QC visual tidak dijalankan otomatis saat training.
# Jika E1 belum dibuat, contoh E0 vs E1 tidak perlu dijalankan.
# Gunakan EDGES dari section graph jika sudah tersedia; jika belum, visualisasi tetap bisa berupa scatter plot.

# Contoh 1: lihat beberapa frame dari E0
# visual_qc_sample(
#     exp_id="E0",
#     fold=1,
#     split="train_fit",
#     sample_idx=0,
#     frames=[0, 10, 20, 40, 63],
#     edges=resolve_qc_edges(),
# )

# Contoh 2: bandingkan E0 dan E1 pada sample/frame yang sama
# visual_compare_experiments(
#     exp_ids=["E0", "E1"],
#     fold=1,
#     split="train_fit",
#     sample_idx=0,
#     frame_idx=20,
#     edges=resolve_qc_edges(),
# )

# Contoh 3: simpan QC 1 label sama untuk seluruh signer.
# save_qc_visual_same_label_all_signers(
#     exp_id="E0",
#     label=0,
#     signers=ALL_SIGNERS,
#     frames=[0, 10, 20, 40, 63],
#     edges=resolve_qc_edges(),
#     axis_mode="stage_fixed",
# )

print("QC visual functions ready. Main mode: same-label all-signers QC with fixed axis.")


## 11. Audit Artifact dan Data Leakage

Audit memeriksa shape, label, jumlah sample, serta memastikan tidak ada overlap antara train, validasi internal, dan test signer.


In [ ]:

# ============================================================
# 11. AUDIT ARTEFAK DAN DATA LEAKAGE CHECK
# ============================================================


def audit_fold_artifacts(exp_id, fold, strict=True):
    fdir = fold_data_dir(exp_id, fold)
    required = [
        fdir / "train_index.csv",
        fdir / "train_fit_C2T64V65.npz",
        fdir / "val_internal_C2T64V65.npz",
        fdir / "test_eval_C2T64V65.npz",
        fdir / "preprocess_config.json",
        fdir / "PREPROCESS_DONE.txt",
    ]

    missing = [str(p) for p in required if not p.exists()]
    if missing and strict:
        raise FileNotFoundError(f"Artefak belum lengkap untuk {exp_id} fold{fold}: {missing}")

    config_match = preprocess_config_matches(exp_id, fold)
    if strict and not config_match:
        raise ValueError(f"{exp_id} fold{fold}: preprocess_config tidak sesuai registry aktif. Rebuild diperlukan.")

    df_train = pd.read_csv(fdir / "train_index.csv") if (fdir / "train_index.csv").exists() else pd.DataFrame()
    df_val = pd.read_csv(fdir / "val_internal_meta.csv") if (fdir / "val_internal_meta.csv").exists() else pd.DataFrame()
    df_test = pd.read_csv(fdir / "test_meta.csv") if (fdir / "test_meta.csv").exists() else pd.DataFrame()

    if len(df_train) and len(df_val) and len(df_test):
        assert_no_leakage(df_train, df_val, df_test, fold)

    train_shape = None
    val_shape = None
    test_shape = None
    train_nan = None
    val_nan = None
    test_nan = None
    train_inf = None
    val_inf = None
    test_inf = None
    if (fdir / "train_fit_C2T64V65.npz").exists():
        d = np.load(fdir / "train_fit_C2T64V65.npz", allow_pickle=True)
        train_shape = tuple(d["X"].shape)
        train_nan = bool(np.isnan(d["X"]).any())
        train_inf = bool(np.isinf(d["X"]).any())
    if (fdir / "val_internal_C2T64V65.npz").exists():
        d = np.load(fdir / "val_internal_C2T64V65.npz", allow_pickle=True)
        val_shape = tuple(d["X"].shape)
        val_nan = bool(np.isnan(d["X"]).any())
        val_inf = bool(np.isinf(d["X"]).any())
    if (fdir / "test_eval_C2T64V65.npz").exists():
        d = np.load(fdir / "test_eval_C2T64V65.npz", allow_pickle=True)
        test_shape = tuple(d["X"].shape)
        test_nan = bool(np.isnan(d["X"]).any())
        test_inf = bool(np.isinf(d["X"]).any())

    if strict:
        for split_name, has_nan, has_inf in [
            ("train_fit", train_nan, train_inf),
            ("val_internal", val_nan, val_inf),
            ("test", test_nan, test_inf),
        ]:
            if has_nan or has_inf:
                raise ValueError(f"{exp_id} fold{fold} {split_name} mengandung NaN/inf.")

    cfg_pre = read_json(fdir / "preprocess_config.json") if (fdir / "preprocess_config.json").exists() else {}

    return {
        "exp_id": exp_id,
        "fold": fold,
        "missing_files": len(missing),
        "config_match": bool(config_match),
        "interp": cfg_pre.get("interp"),
        "norm": cfg_pre.get("norm"),
        "augment": cfg_pre.get("augment"),
        "target_T": cfg_pre.get("target_T"),
        "train_n": int(len(df_train)),
        "train_shape": train_shape,
        "val_shape": val_shape,
        "test_shape": test_shape,
        "train_has_nan": train_nan,
        "val_has_nan": val_nan,
        "test_has_nan": test_nan,
        "train_has_inf": train_inf,
        "val_has_inf": val_inf,
        "test_has_inf": test_inf,
        "train_signers": ",".join(map(str, sorted(df_train["signer_id"].unique()))) if len(df_train) else "",
        "test_signer": fold,
    }


def audit_experiment_artifacts(exp_id, strict=True):
    rows = [audit_fold_artifacts(exp_id, fold, strict=strict) for fold in TEST_SIGNERS]
    df = pd.DataFrame(rows)
    display(df)
    return df

print("Audit functions ready.")


## 12. Definisi Graph ResGCN

Pose direpresentasikan sebagai graph 65 node dengan koneksi anatomi tangan, kepala, dan tubuh bagian atas yang digunakan oleh graph convolution.


In [ ]:
# === [4] GRAPH DEFINITION ===

V = 65
MAX_HOP = 2  # paper: max graph distance = 2:contentReference[oaicite:5]{index=5}

def mp_hand_edges(offset=0):
    # Mediapipe hand: 0 wrist, 1-4 thumb, 5-8 index, 9-12 middle, 13-16 ring, 17-20 pinky
    e = []
    chains = [
        (0,1,2,3,4),
        (0,5,6,7,8),
        (0,9,10,11,12),
        (0,13,14,15,16),
        (0,17,18,19,20)
    ]
    for c in chains:
        for a,b in zip(c[:-1], c[1:]):
            e.append((a+offset, b+offset))
            e.append((b+offset, a+offset))
    # palm connections between MCPs
    palm = [(5,9),(9,13),(13,17),(17,5)]
    for a,b in palm:
        e.append((a+offset, b+offset))
        e.append((b+offset, a+offset))
    return e

def build_edges_65():
    edges = []

    # Left hand (0-20), Right hand (21-41)
    edges += mp_hand_edges(offset=0)
    edges += mp_hand_edges(offset=21)

    # Head (42-52): [NOSE, LEyeIn, LEye, LEyeOut, REyeIn, REye, REyeOut, LEar, REar, MouthL, MouthR]
    NOSE = 42
    LEYE = 44
    REYE = 46
    LEAR = 49
    REAR = 50
    ML = 51
    MR = 52

    head_pairs = [
        (NOSE, LEYE), (NOSE, REYE),
        (LEYE, LEAR), (REYE, REAR),
        (NOSE, ML), (NOSE, MR),
        (ML, MR)
    ]
    for a,b in head_pairs:
        edges += [(a,b),(b,a)]

    # Body (53-64):
    # 53 NECK
    # 54 L_SHOULDER, 55 R_SHOULDER
    # 56 L_ELBOW, 57 R_ELBOW
    # 58 L_WRIST, 59 R_WRIST
    # 60 L_HIP, 61 R_HIP
    # 62 MID_HIP
    # 63 L_INDEX (pose idx 19), 64 R_INDEX (pose idx 20) (opsional bantu “direction” tangan)
    NECK=53; LS=54; RS=55; LE=56; RE=57; LW=58; RW=59; LH=60; RH=61; MH=62; LIDX=63; RIDX=64

    body_pairs = [
        (NECK, LS), (NECK, RS),
        (LS, LE), (RS, RE),
        (LE, LW), (RE, RW),
        (LH, MH), (RH, MH),
        (LW, LIDX), (RW, RIDX),
        # connect head to body
        (NECK, NOSE),
    ]
    for a,b in body_pairs:
        edges += [(a,b),(b,a)]

    # connect wrists to hand wrists
    LH_WRIST_NODE = 0
    RH_WRIST_NODE = 21
    edges += [(LW, LH_WRIST_NODE), (LH_WRIST_NODE, LW)]
    edges += [(RW, RH_WRIST_NODE), (RH_WRIST_NODE, RW)]

    return edges

EDGES = build_edges_65()

def adjacency_from_edges(num_nodes, edges):
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    for i,j in edges:
        A[i,j] = 1.0
    return A

A0 = adjacency_from_edges(V, EDGES)

def hop_distance(A, max_hop=2):
    # Floyd-warshall-ish via BFS on adjacency
    dist = np.full((V,V), np.inf, dtype=np.float32)
    for i in range(V):
        dist[i,i]=0
        frontier=[i]
        for d in range(1, max_hop+1):
            nxt=[]
            for u in frontier:
                vs = np.where(A[u] > 0)[0].tolist()
                for v in vs:
                    if dist[i,v] > d:
                        dist[i,v] = d
                        nxt.append(v)
            frontier = list(set(nxt))
    return dist

DIST = hop_distance(A0, max_hop=MAX_HOP)

def normalize_digraph(A):
    Dl = np.sum(A, axis=0)
    Dn = np.zeros((V, V), dtype=np.float32)
    for i in range(V):
        if Dl[i] > 0:
            Dn[i,i] = Dl[i] ** (-1)
    return A @ Dn

K = MAX_HOP + 1
A_stack = np.zeros((K, V, V), dtype=np.float32)

# 1) add self-loops in hop-0
I = np.eye(V, dtype=np.float32)

for hop in range(K):
    if hop == 0:
        A_hop = I
    else:
        A_hop = (DIST == hop).astype(np.float32)
    A_stack[hop] = normalize_digraph(A_hop)

A_torch = torch.tensor(A_stack, dtype=torch.float32, device=DEVICE)

print("A_stack shape:", A_stack.shape)  # (K,V,V)


## 13. Dataset dan Augmentasi Train-only

Augmentasi dilakukan on-the-fly hanya pada `train_fit`. Validasi internal dan test tidak menerima augmentasi.


In [ ]:

# ============================================================
# 13. DATASET DAN AUGMENTASI TRAIN-ONLY
# ============================================================


def valid_mask_ctv_from_coords(X):
    """Fallback mask valid berdasarkan rule: (x != 0) or (y != 0)."""
    return (X[0] != 0) | (X[1] != 0)  # (T,V)


def apply_random_rotation_valid_centroid(X, valid_mask=None, max_angle=10, p=0.5):
    """
    Rotasi kecil dengan pusat valid_joint_centroid.
    X shape: (C,T,V). valid_mask shape: (T,V).
    """
    if np.random.rand() > p:
        return X
    X_aug = X.copy()
    if valid_mask is None:
        valid_mask = valid_mask_ctv_from_coords(X_aug)
    valid_mask = valid_mask.astype(bool)
    if not valid_mask.any():
        return X_aug

    angle = np.random.uniform(-max_angle, max_angle)
    rad = math.radians(angle)
    c, s = math.cos(rad), math.sin(rad)

    coords = np.stack([X_aug[0][valid_mask], X_aug[1][valid_mask]], axis=1)
    center = coords.mean(axis=0)
    xy = coords - center
    rot = np.empty_like(xy)
    rot[:, 0] = xy[:, 0] * c - xy[:, 1] * s
    rot[:, 1] = xy[:, 0] * s + xy[:, 1] * c
    rot = rot + center

    X_aug[0][valid_mask] = rot[:, 0]
    X_aug[1][valid_mask] = rot[:, 1]
    return X_aug.astype(np.float32)


def apply_spatial_jitter_valid(X, valid_mask=None, sigma=0.015, p=0.5):
    """Spatial jitter Gaussian hanya pada joint valid."""
    if np.random.rand() > p:
        return X
    X_aug = X.copy()
    if valid_mask is None:
        valid_mask = valid_mask_ctv_from_coords(X_aug)
    valid_mask = valid_mask.astype(bool)
    if not valid_mask.any():
        return X_aug

    noise = np.random.normal(loc=0.0, scale=sigma, size=X_aug.shape).astype(np.float32)
    X_aug[0][valid_mask] += noise[0][valid_mask]
    X_aug[1][valid_mask] += noise[1][valid_mask]
    return X_aug.astype(np.float32)


def apply_pose_spatial_aug_v1(X, valid_mask=None):
    """Urutan augmentasi final: rotasi kecil lalu spatial jitter."""
    X = apply_random_rotation_valid_centroid(
        X,
        valid_mask=valid_mask,
        max_angle=10,
        p=0.5,
    )
    X = apply_spatial_jitter_valid(
        X,
        valid_mask=valid_mask,
        sigma=0.015,
        p=0.5,
    )
    return X.astype(np.float32)


class PoseTrainDatasetFromIndexNPZ(Dataset):
    """
    Dataset training dari train_index.csv.
    Augmentasi hanya aktif jika augment=True dan hanya digunakan pada Dataset training.
    """
    def __init__(self, index_csv, augment=False):
        self.df = pd.read_csv(index_csv)
        assert "sample_path" in self.df.columns, "train_index.csv harus punya kolom sample_path"
        assert "label" in self.df.columns, "train_index.csv harus punya kolom label"
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        y = int(r["label"])
        p = Path(str(r["sample_path"]))
        d = np.load(p, allow_pickle=True)
        X = d["X"].astype(np.float32)  # (C,T,V)
        if "valid_mask" in d:
            valid_mask = d["valid_mask"].astype(bool)  # (T,V)
        else:
            valid_mask = valid_mask_ctv_from_coords(X)

        if self.augment:
            X = apply_pose_spatial_aug_v1(X, valid_mask=valid_mask)

        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long)


class PoseTrainDatasetFromPack(Dataset):
    """Dataset training dari train_fit_C2T64V65.npz. Augmentasi tetap train-only."""
    def __init__(self, pack_npz: Path, augment=False):
        d = np.load(pack_npz, allow_pickle=True)
        self.X = d["X"].astype(np.float32)
        self.y = d["y"].astype(np.int64)
        self.video_id = d["video_id"] if "video_id" in d else np.array([str(i) for i in range(len(self.y))], dtype=object)
        self.valid_mask = d["valid_mask"].astype(bool) if "valid_mask" in d else None
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        X = self.X[idx].copy()
        y = int(self.y[idx])
        valid_mask = self.valid_mask[idx] if self.valid_mask is not None else valid_mask_ctv_from_coords(X)
        if self.augment:
            X = apply_pose_spatial_aug_v1(X, valid_mask=valid_mask)
        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long)


class PoseEvalDatasetFromPack(Dataset):
    """Dataset evaluasi dari pack NPZ. Tidak ada augmentasi untuk val_internal atau test."""
    def __init__(self, pack_npz: Path):
        d = np.load(pack_npz, allow_pickle=True)
        self.X = d["X"].astype(np.float32)
        self.y = d["y"].astype(np.int64)
        self.video_id = d["video_id"] if "video_id" in d else np.array([str(i) for i in range(len(self.y))], dtype=object)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)

print("Dataset and augmentation functions ready.")


## 14. Model ResGCN_N39_R4

Arsitektur model dipertahankan sama pada E0–E7; hanya konfigurasi preprocessing yang berubah.


In [ ]:
# === [6] MODEL ===

class GraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, K):
        super().__init__()
        self.K = K
        self.conv = nn.Conv2d(in_channels, out_channels * K, kernel_size=1)

    def forward(self, x, A):  # x: [N,C,T,V], A: [K,V,V]
        N, C, T, V = x.shape
        x = self.conv(x)  # [N, out*K, T, V]
        x = x.view(N, self.K, -1, T, V)  # [N,K,out,T,V]
        out = 0
        for k in range(self.K):
            # einsum over V: (N,out,T,V) = (N,out,T,V) * (V,V)
            out = out + torch.einsum('nctv,vw->nctw', x[:,k], A[k])
        return out

class BasicSTGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, t_kernel=9, stride=1, dropout=0.0):
        super().__init__()
        K = A.shape[0]
        self.A = A

        self.gcn = GraphConv(in_c, out_c, K)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, kernel_size=(t_kernel,1), padding=((t_kernel-1)//2,0), stride=(stride,1))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if (in_c == out_c) and (stride == 1):
            self.res = nn.Identity()
        else:
            self.res = nn.Conv2d(in_c, out_c, kernel_size=1, stride=(stride,1))

    def forward(self, x):
        res = self.res(x)
        x = self.gcn(x, self.A)
        x = F.relu(self.bn1(x))
        x = self.tcn(x)
        x = self.bn2(x)
        x = self.drop(x)
        x = F.relu(x + res)
        return x

class BottleneckSTGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, R=4, t_kernel=9, stride=1, dropout=0.0):
        super().__init__()
        K = A.shape[0]
        self.A = A
        mid = max(1, out_c // R)

        # spatial bottleneck (no temporal stride here)
        self.s_reduce = nn.Conv2d(in_c, mid, kernel_size=1)
        self.s_gcn = GraphConv(mid, mid, K)
        self.s_expand = nn.Conv2d(mid, out_c, kernel_size=1)
        self.s_bn = nn.BatchNorm2d(out_c)

        # residual for spatial stage: MUST NOT change T
        if in_c == out_c:
            self.res_s = nn.Identity()
        else:
            self.res_s = nn.Conv2d(in_c, out_c, kernel_size=1, stride=(1,1))

        # temporal bottleneck (stride applies here)
        self.t_reduce = nn.Conv2d(out_c, mid, kernel_size=1)
        self.t_conv = nn.Conv2d(
            mid, mid,
            kernel_size=(t_kernel,1),
            padding=((t_kernel-1)//2,0),
            stride=(stride,1)
        )
        self.t_expand = nn.Conv2d(mid, out_c, kernel_size=1)
        self.t_bn = nn.BatchNorm2d(out_c)

        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # residual for temporal stage: MUST match temporal stride
        if stride == 1:
            self.res_t = nn.Identity()
        else:
            self.res_t = nn.Conv2d(out_c, out_c, kernel_size=1, stride=(stride,1))

    def forward(self, x):
        # ---- spatial stage (T unchanged) ----
        res_s = self.res_s(x)

        y = self.s_reduce(x)
        y = self.s_gcn(y, self.A)
        y = self.s_expand(y)
        y = F.relu(self.s_bn(y) + res_s)

        # ---- temporal stage (T may downsample) ----
        y2 = self.t_reduce(y)
        y2 = self.t_conv(y2)
        y2 = self.t_expand(y2)
        y2 = self.t_bn(y2)
        y2 = self.drop(y2)

        res_t = self.res_t(y)
        out = F.relu(y2 + res_t)
        return out

class ResGCN_N39_R4(nn.Module):
    """
    Mengikuti ringkasan Table 4 di paper (ResGCN-n39, R=4).
    Output temporal downsample: 64 -> 32 -> 16 pada bottleneck block3 & block5.
    """
    def __init__(self, num_classes, A, R=4, t_kernel=9, dropout=0.1):
        super().__init__()
        self.A = A
        self.num_classes = num_classes

        # input BN (common ST-GCN trick): BN over (C*V)
        self.data_bn = nn.BatchNorm1d(2 * 65)

        # Basic Block: out 64
        self.basic = BasicSTGCNBlock(2, 64, A, t_kernel=t_kernel, stride=1, dropout=dropout)

        # 6 bottleneck blocks (paper: N39 has 6 bottleneck blocks)
        self.b1 = BottleneckSTGCNBlock(64, 64, A, R=R, t_kernel=t_kernel, stride=1, dropout=dropout)
        self.b2 = BottleneckSTGCNBlock(64, 32, A, R=R, t_kernel=t_kernel, stride=1, dropout=dropout)
        self.b3 = BottleneckSTGCNBlock(32, 128, A, R=R, t_kernel=t_kernel, stride=2, dropout=dropout)  # T/2
        self.b4 = BottleneckSTGCNBlock(128, 128, A, R=R, t_kernel=t_kernel, stride=1, dropout=dropout)
        self.b5 = BottleneckSTGCNBlock(128, 256, A, R=R, t_kernel=t_kernel, stride=2, dropout=dropout) # T/2
        self.b6 = BottleneckSTGCNBlock(256, 256, A, R=R, t_kernel=t_kernel, stride=1, dropout=dropout)

        # Global Avg Pool
        self.gap = nn.AdaptiveAvgPool2d((1,1))

        # embedding size = num_classes (paper)
        self.fc1 = nn.Linear(256, num_classes)
        self.fc2 = nn.Linear(num_classes, num_classes)

    def forward(self, x):  # x [N,2,64,65]
        N, C, T, V = x.shape

        # data_bn expects [N, C*V, T]
        x = x.permute(0,3,1,2).contiguous().view(N, V*C, T)
        x = self.data_bn(x)
        x = x.view(N, V, C, T).permute(0,2,3,1).contiguous()  # [N,C,T,V]

        x = self.basic(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        x = self.b6(x)

        x = self.gap(x).view(N, -1)  # [N,256]
        x = self.fc2(F.relu(self.fc1(x)))
        return x


## 15. Training, Evaluation, dan Checkpointing

Training menggunakan Adam, CyclicLR, cross-entropy loss, dan early stopping berdasarkan validation loss. Checkpoint memungkinkan resume setelah sesi Colab terputus.


In [ ]:
# ============================================================
# 15. TRAINING, EVALUATION, DAN CHECKPOINTING
# ============================================================


def topk_accuracy(logits, y, ks=(1, 5, 10)):
    with torch.no_grad():
        maxk = max(ks)
        _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)
        correct = pred.eq(y.view(-1, 1))
        out = {}
        for k in ks:
            out[k] = correct[:, :k].any(dim=1).float().mean().item()
        return out


def run_eval(model, loader, ks=(1, 5, 10)):
    model.eval()
    total_loss = 0.0
    all_logits, all_y = [], []
    ce = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            logits = model(X)
            loss = ce(logits, y)
            total_loss += loss.item() * y.size(0)
            all_logits.append(logits.detach().cpu())
            all_y.append(y.detach().cpu())
    logits = torch.cat(all_logits, dim=0)
    y = torch.cat(all_y, dim=0)
    accs = topk_accuracy(logits, y, ks=ks)
    return total_loss / len(y), accs, logits.numpy(), y.numpy()


def train_one_epoch(model, loader, optimizer, scheduler=None, desc="train"):
    model.train()
    ce = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    pbar = tqdm(loader, desc=desc, leave=False)
    for X, y in pbar:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X)
        loss = ce(logits, y)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n += bs
        pbar.set_postfix(loss=total_loss / total_n, acc=total_correct / total_n, lr=optimizer.param_groups[0]["lr"])

    return total_loss / total_n, total_correct / total_n


def fmt_hms(sec: float) -> str:
    sec = int(sec)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def save_csv_atomic(df: pd.DataFrame, final_path: Path):
    final_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = final_path.with_suffix(final_path.suffix + f".{int(time.time())}.tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(final_path)


def fold_training_complete(exp_id, fold):
    """Fold selesai hanya jika DONE dan seluruh artefak evaluasi utama tersedia."""
    fdir = fold_run_dir(exp_id, fold)
    required_files = [
        fdir / "DONE.txt",
        fdir / f"metrics_fold{fold}.csv",
        fdir / f"confusion_matrix_fold{fold}.csv",
        fdir / f"classification_report_fold{fold}.csv",
        fdir / f"test_predictions_fold{fold}.csv",
        fdir / "runtime_log.json",
        fdir / "best_checkpoint.pt",
    ]
    return all(p.exists() for p in required_files)


def experiment_complete(exp_id):
    return all(fold_training_complete(exp_id, fold) for fold in TEST_SIGNERS)


def fit_one_fold(exp_id, fold, force=False):
    cfg = EXPERIMENTS[exp_id]
    fdata = fold_data_dir(exp_id, fold)
    frun = fold_run_dir(exp_id, fold)
    frun.mkdir(parents=True, exist_ok=True)

    done_flag = frun / "DONE.txt"
    if fold_training_complete(exp_id, fold) and not force:
        print(f"[SKIP TRAIN] {exp_id} fold{fold}: DONE dan artefak evaluasi lengkap")
        metrics_path = frun / f"metrics_fold{fold}.csv"
        return pd.read_csv(metrics_path).iloc[0].to_dict()

    if done_flag.exists() and not fold_training_complete(exp_id, fold) and not force:
        print(f"[RECHECK TRAIN] {exp_id} fold{fold}: DONE.txt ada, tetapi artefak evaluasi belum lengkap. Fold akan diproses ulang/resume.")

    if force and frun.exists():
        shutil.rmtree(frun)
        frun.mkdir(parents=True, exist_ok=True)

    required_data = [
        fdata / "train_index.csv",
        fdata / "train_fit_C2T64V65.npz",
        fdata / "val_internal_C2T64V65.npz",
        fdata / "test_eval_C2T64V65.npz",
        fdata / "PREPROCESS_DONE.txt",
    ]
    missing = [str(p) for p in required_data if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Data fold belum lengkap untuk {exp_id} fold{fold}: {missing}")

    config_fold = {
        "exp_id": exp_id,
        "seed": int(SEED),
        "name": cfg["name"],
        "fold": fold,
        "test_signer": fold,
        "interp": cfg["interp"],
        "norm": cfg["norm"],
        "augment": cfg["augment"],
        "augment_config": AUGMENT_CONFIG if cfg["augment"] else None,
        "hyperparams": HYPERPARAMS,
        "data_dir": str(fdata),
        "run_dir": str(frun),
    }
    write_json(frun / "config_fold.json", config_fold)

    start_wall = datetime.now()
    start_time = start_wall.isoformat(timespec="seconds")
    t_all0 = time.time()

    train_pack_path = fdata / "train_fit_C2T64V65.npz"
    if train_pack_path.exists():
        train_ds = PoseTrainDatasetFromPack(train_pack_path, augment=cfg["augment"])
    else:
        # Fallback konservatif untuk artefak lama sebelum revisi train pack.
        train_ds = PoseTrainDatasetFromIndexNPZ(fdata / "train_index.csv", augment=cfg["augment"])
    val_ds = PoseEvalDatasetFromPack(fdata / "val_internal_C2T64V65.npz")
    test_ds = PoseEvalDatasetFromPack(fdata / "test_eval_C2T64V65.npz")

    train_loader = DataLoader(train_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

    model = ResGCN_N39_R4(
        num_classes=NUM_CLASSES,
        A=A_torch,
        R=HYPERPARAMS["R"],
        t_kernel=HYPERPARAMS["t_kernel"],
        dropout=HYPERPARAMS["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=HYPERPARAMS["lr"])
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr=HYPERPARAMS["lr"] / 10,
        max_lr=HYPERPARAMS["lr"],
        step_size_up=max(1, len(train_loader) * 5),
        cycle_momentum=False,
    )

    best_ckpt = frun / "best_checkpoint.pt"
    last_ckpt = frun / "last_checkpoint.pt"
    log_csv = frun / "training_log.csv"

    start_epoch = 1
    best_val_loss = float("inf")
    best_val_top1 = 0.0
    best_epoch = 0
    no_improve = 0
    elapsed_seconds = 0.0
    log_rows = []

    if log_csv.exists():
        try:
            log_rows = pd.read_csv(log_csv).to_dict("records")
        except Exception:
            log_rows = []

    if last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optim"])
        scheduler.load_state_dict(ckpt["sched"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
        best_val_top1 = float(ckpt.get("best_val_top1", 0.0))
        best_epoch = int(ckpt.get("best_epoch", 0))
        no_improve = int(ckpt.get("no_improve", 0))
        elapsed_seconds = float(ckpt.get("elapsed_seconds", 0.0))
        print(f"[RESUME] {exp_id} fold{fold} from epoch {start_epoch} | best_val_loss={best_val_loss:.4f} | elapsed={fmt_hms(elapsed_seconds)}")

    for ep in range(start_epoch, HYPERPARAMS["epochs"] + 1):
        seed_all(SEED + fold * 1000 + ep)
        ep_start = time.time()

        t_train0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, desc=f"{exp_id} fold{fold} ep{ep:03d}")
        t_train = time.time() - t_train0

        t_val0 = time.time()
        va_loss, va_accs, _, _ = run_eval(model, val_loader, ks=(1, 5, 10))
        t_val = time.time() - t_val0

        ep_seconds = time.time() - ep_start
        elapsed_seconds += ep_seconds

        improved = va_loss < best_val_loss - HYPERPARAMS["min_delta"]
        if improved:
            best_val_loss = float(va_loss)
            best_val_top1 = float(va_accs[1])
            best_epoch = ep
            no_improve = 0
            torch.save({
                "model": model.state_dict(),
                "epoch": ep,
                "best_val_loss": best_val_loss,
                "best_val_top1": best_val_top1,
            }, best_ckpt)
        else:
            no_improve += 1

        row = {
            "epoch": ep,
            "train_loss": tr_loss,
            "train_top1": tr_acc,
            "val_loss": va_loss,
            "val_top1": va_accs[1],
            "val_top5": va_accs[5],
            "val_top10": va_accs[10],
            "lr": optimizer.param_groups[0]["lr"],
            "train_sec": t_train,
            "val_sec": t_val,
            "epoch_sec": ep_seconds,
            "elapsed_sec": elapsed_seconds,
            "best_val_loss": best_val_loss,
            "best_val_top1": best_val_top1,
            "best_epoch": best_epoch,
            "no_improve": no_improve,
        }
        log_rows.append(row)
        pd.DataFrame(log_rows).to_csv(log_csv, index=False)

        if (ep % HYPERPARAMS["save_every"] == 0) or (ep == HYPERPARAMS["epochs"]):
            torch.save({
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "epoch": ep,
                "best_val_loss": best_val_loss,
                "best_val_top1": best_val_top1,
                "best_epoch": best_epoch,
                "no_improve": no_improve,
                "elapsed_seconds": elapsed_seconds,
            }, last_ckpt)

        print(f"[{exp_id} fold{fold}] ep {ep:03d}/{HYPERPARAMS['epochs']} | tr_loss {tr_loss:.4f} | tr@1 {tr_acc:.3f} | "
              f"va_loss {va_loss:.4f} | va@1 {va_accs[1]:.3f} | best_loss {best_val_loss:.4f} "
              f"(ep{best_epoch}) | no_improve {no_improve} | elapsed {fmt_hms(elapsed_seconds)}")

        if HYPERPARAMS["early_stop"] and ep >= HYPERPARAMS["warmup_epochs"] and no_improve >= HYPERPARAMS["patience"]:
            print(f"[EARLY STOP] {exp_id} fold{fold} at epoch {ep}")
            torch.save({
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "epoch": ep,
                "best_val_loss": best_val_loss,
                "best_val_top1": best_val_top1,
                "best_epoch": best_epoch,
                "no_improve": no_improve,
                "elapsed_seconds": elapsed_seconds,
            }, last_ckpt)
            break

    if not best_ckpt.exists():
        raise RuntimeError(f"best_checkpoint.pt tidak ditemukan untuk {exp_id} fold{fold}. Training mungkin gagal sebelum validasi.")

    t_test0 = time.time()
    ckpt = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    te_loss, te_accs, te_logits, te_y = run_eval(model, test_loader, ks=(1, 5, 10))
    test_seconds = time.time() - t_test0

    y_pred = te_logits.argmax(1)

    cm = confusion_matrix(te_y, y_pred, labels=list(range(NUM_CLASSES)))
    pd.DataFrame(cm).to_csv(frun / f"confusion_matrix_fold{fold}.csv", index=False)

    rep = classification_report(te_y, y_pred, digits=4, output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(frun / f"classification_report_fold{fold}.csv")

    pred_df = pd.DataFrame({
        "video_id": test_ds.video_id.astype(str),
        "y_true": te_y.astype(int),
        "y_pred": y_pred.astype(int),
    })
    for cls in range(NUM_CLASSES):
        pred_df[f"logit_{cls}"] = te_logits[:, cls]
    pred_df.to_csv(frun / f"test_predictions_fold{fold}.csv", index=False)

    total_seconds = time.time() - t_all0
    end_time = datetime.now().isoformat(timespec="seconds")

    metrics = {
        "exp_id": exp_id,
        "seed": int(SEED),
        "name": cfg["name"],
        "fold": fold,
        "test_signer": fold,
        "interp": cfg["interp"],
        "norm": cfg["norm"],
        "augment": cfg["augment"],
        "selection_metric": HYPERPARAMS["selection_metric"],
        "best_val_loss": float(best_val_loss),
        "best_val_top1": float(best_val_top1),
        "best_epoch": int(best_epoch),
        "test_loss": float(te_loss),
        "test_top1": float(te_accs[1]),
        "test_top5": float(te_accs[5]),
        "test_top10": float(te_accs[10]),
        "elapsed_train_val_seconds": float(elapsed_seconds),
        "elapsed_train_val_hms": fmt_hms(elapsed_seconds),
        "test_seconds": float(test_seconds),
        "test_hms": fmt_hms(test_seconds),
        "total_seconds_est": float(elapsed_seconds + test_seconds),
        "total_hms_est": fmt_hms(elapsed_seconds + test_seconds),
        "out_dir": str(frun),
    }
    pd.DataFrame([metrics]).to_csv(frun / f"metrics_fold{fold}.csv", index=False)

    runtime_log = {
        "experiment": exp_id,
        "seed": int(SEED),
        "fold": fold,
        "test_signer": fold,
        "start_time": start_time,
        "end_time": end_time,
        "duration_minutes": float(total_seconds / 60.0),
        "device": DEVICE,
        "gpu_type": GPU_TYPE,
        "best_epoch": int(best_epoch),
        "best_val_top1": float(best_val_top1),
        "test_top1": float(te_accs[1]),
        "test_top5": float(te_accs[5]),
        "test_top10": float(te_accs[10]),
    }
    write_json(frun / "runtime_log.json", runtime_log)

    # DONE ditulis paling akhir agar fold hanya dianggap selesai setelah evaluasi selesai.
    done_flag.write_text(datetime.now().isoformat(timespec="seconds"))
    print(f"[TRAIN DONE] {exp_id} fold{fold} | test@1={te_accs[1]:.4f} | total~={metrics['total_hms_est']}")
    return metrics

print("Training functions ready.")


## 16. Menjalankan Eksperimen

Urutan eksekusi: build artifact → validasi konfigurasi → QC opsional → training empat fold → rekap single-seed.


### 16.1 Build Data Artifact

Cell ini membangun artifact preprocessing untuk `ACTIVE_EXPERIMENTS`. Jika artifact fold sudah lengkap dan `FORCE_REBUILD_DATA=False`, preprocessing fold akan dilewati melalui mekanisme resume.

In [ ]:

# ============================================================
# 16.1 BUILD DATA ARTIFACT
# ============================================================

built_experiments = []

if AUTO_BUILD_DATA:
    for exp_id in ACTIVE_EXPERIMENTS:
        print("=" * 90)
        print(f"[BUILD DATA] EXPERIMENT {exp_id}: {EXPERIMENTS[exp_id]['name']}")
        print(EXPERIMENTS[exp_id])
        build_artifacts_for_experiment(exp_id, force=FORCE_REBUILD_DATA)
        audit_experiment_artifacts(exp_id, strict=True)
        built_experiments.append(exp_id)
else:
    print("AUTO_BUILD_DATA=False, build artifact dilewati.")


### 16.1B Validasi Konfigurasi Preprocessing Final

In [ ]:

# ============================================================
# 16.1B VALIDASI KONFIGURASI PREPROCESSING FINAL
# ============================================================

for exp_id in ACTIVE_EXPERIMENTS:
    for fold in TEST_SIGNERS:
        fdir = fold_data_dir(exp_id, fold)
        config_path = fdir / "preprocess_config.json"

        if not config_path.exists():
            print(f"[WARN] Config belum ada: {config_path}")
            continue

        cfg_pre = read_json(config_path)
        print(
            f"{exp_id} fold{fold} | "
            f"interp={cfg_pre.get('interp')} | "
            f"norm={cfg_pre.get('norm')} | "
            f"augment={cfg_pre.get('augment')} | "
            f"target_T={cfg_pre.get('target_T')} | "
            f"train_n={cfg_pre.get('train_n')} | "
            f"val_n={cfg_pre.get('val_internal_n')} | "
            f"test_n={cfg_pre.get('test_n')}"
        )

        expected = EXPERIMENTS[exp_id]
        for key in ["interp", "norm", "augment"]:
            if bool(cfg_pre.get(key)) != bool(expected[key]):
                raise AssertionError(f"{exp_id} fold{fold}: {key} tidak sesuai registry eksperimen")
        if int(cfg_pre.get("target_T")) != int(TARGET_T):
            raise AssertionError(f"{exp_id} fold{fold}: target_T tidak sesuai")

print("Validasi selesai: konfigurasi preprocessing aktif sesuai registry eksperimen.")


### 16.2 QC Missing Keypoint Artifact

Cell ini menyimpan laporan `missing_rate_report_<EXP_ID>.csv` untuk eksperimen aktif. QC ini hanya membaca artifact `.npz` dan tidak menjadi syarat training.

In [ ]:

# ============================================================
# 16.2 QC MISSING KEYPOINT ARTIFACT
# ============================================================

if AUTO_BUILD_DATA:
    for exp_id in ACTIVE_EXPERIMENTS:
        print(f"[QC MISSING] Missing keypoint audit for {exp_id}")
        save_missing_rate_report(exp_id)
else:
    print("AUTO_BUILD_DATA=False, QC missing keypoint otomatis dilewati.")

# Contoh manual setelah E0 dan E1 selesai preprocessing:
# compare_missing_rate_between_experiments(["E0", "E1"])


### 16.3 QC Visual Artifact (Opsional)

QC visual dapat disimpan untuk memeriksa pose pada label yang sama di beberapa signer. Default `AUTO_SAVE_QC_VISUAL=False` agar repeated run tidak menghasilkan gambar duplikat. Aktifkan hanya saat diperlukan untuk audit.


In [ ]:
# ============================================================
# 16.3 SIMPAN QC VISUAL ARTIFACT
# ============================================================

# AUTO_SAVE_QC_VISUAL didefinisikan pada konfigurasi run.
# QC visual bersifat diagnostik dan tidak memengaruhi training.

# Mode utama: satu label yang sama, satu sample per signer 0-4.
# Karena signer 0 hanya memiliki sebagian label, pilih label yang tersedia untuk semua signer.
# Untuk dataset saat ini, pilihan aman umumnya label 0-11.
QC_VISUAL_MODE = "same_label_all_signers"
QC_VISUAL_LABEL = 0
QC_VISUAL_SIGNERS = ALL_SIGNERS
QC_VISUAL_FRAMES = [0, 10, 20, 40, 63]
QC_VISUAL_PREFERRED_SPLIT = "train_fit"

# Fixed axis mencegah Matplotlib melakukan auto-zoom per subplot.
# stage_fixed = raw dibandingkan dengan raw, artifact dengan artifact, augmented dengan augmented.
QC_VISUAL_AXIS_MODE = "stage_fixed"

print("Common labels untuk semua signer:", get_common_labels_for_signers(QC_VISUAL_SIGNERS))
print("QC visual default menyimpan 5 frame representatif:", QC_VISUAL_FRAMES)
print("QC visual mengambil 1 sample per signer pada label yang sama.")
print("QC visual axis mode:", QC_VISUAL_AXIS_MODE)

if AUTO_SAVE_QC_VISUAL:
    for exp_id in ACTIVE_EXPERIMENTS:
        print(f"[QC VISUAL] Saving same-label all-signers QC for {exp_id}")
        save_qc_visual_same_label_all_signers(
            exp_id=exp_id,
            label=QC_VISUAL_LABEL,
            signers=QC_VISUAL_SIGNERS,
            frames=QC_VISUAL_FRAMES,
            edges=resolve_qc_edges(),
            preferred_split=QC_VISUAL_PREFERRED_SPLIT,
            include_raw=True,
            include_augmented_preview=True,
            axis_mode=QC_VISUAL_AXIS_MODE,
            preview=True,
        )
else:
    print("AUTO_SAVE_QC_VISUAL=False, penyimpanan QC visual dilewati.")
    print("Untuk menyimpan QC visual ke Drive, ubah AUTO_SAVE_QC_VISUAL=True pada cell ini.")

# Contoh manual:
# save_qc_visual_same_label_all_signers(
#     exp_id="E2",
#     label=0,
#     signers=ALL_SIGNERS,
#     frames=[0, 10, 20, 40, 63],
#     edges=resolve_qc_edges(),
#     axis_mode="stage_fixed",
#     preview=True,
# )


### 16.4 Training Fold

Cell ini menjalankan training untuk `ACTIVE_EXPERIMENTS` hanya jika `AUTO_TRAIN=True`. Jika fold sudah lengkap dan `FORCE_RERUN_TRAINING=False`, fold akan dilewati melalui completion check.

In [ ]:

# ============================================================
# 16.4 TRAINING FOLD
# ============================================================

all_run_results = []

if AUTO_TRAIN:
    for exp_id in ACTIVE_EXPERIMENTS:
        print("=" * 90)
        print(f"[TRAIN] EXPERIMENT {exp_id}: {EXPERIMENTS[exp_id]['name']}")
        for fold in TEST_SIGNERS:
            result = fit_one_fold(exp_id, fold, force=FORCE_RERUN_TRAINING)
            all_run_results.append(result)
else:
    print("AUTO_TRAIN=False, training dilewati.")

if all_run_results:
    display(pd.DataFrame(all_run_results))


### 16.5 Ringkasan Output Run Aktif

Cell ini menampilkan ringkasan keberadaan output utama untuk eksperimen aktif agar output run tidak menumpuk pada satu cell.

In [ ]:

# ============================================================
# 16.5 RINGKASAN OUTPUT RUN AKTIF
# ============================================================

rows = []
for exp_id in ACTIVE_EXPERIMENTS:
    for fold in TEST_SIGNERS:
        rows.append({
            "exp_id": exp_id,
            "fold": fold,
            "preprocess_complete": preprocess_fold_complete(exp_id, fold),
            "training_complete": fold_training_complete(exp_id, fold),
            "data_dir": str(fold_data_dir(exp_id, fold)),
            "run_dir": str(fold_run_dir(exp_id, fold)),
            "qc_dir": str(qc_visual_dir(exp_id)),
        })

df_active_status = pd.DataFrame(rows)
display(df_active_status)


## 17. Rekap Hasil per Fold

Metrik setiap test signer/fold dikumpulkan ke satu tabel untuk eksperimen aktif.


In [ ]:

# ============================================================
# 17. REKAP HASIL PER FOLD
# ============================================================


def summarize_experiment(exp_id):
    rows = []
    for fold in TEST_SIGNERS:
        metrics_path = fold_run_dir(exp_id, fold) / f"metrics_fold{fold}.csv"
        if fold_training_complete(exp_id, fold) and metrics_path.exists():
            rows.append(pd.read_csv(metrics_path).iloc[0].to_dict())
        elif metrics_path.exists():
            print(f"[WARN] metrics ada, tetapi completion check fold belum lengkap: {exp_id} fold{fold}")
            rows.append(pd.read_csv(metrics_path).iloc[0].to_dict())
        else:
            print(f"[WARN] metrics belum ada: {metrics_path}")

    if not rows:
        print(f"Belum ada metrics untuk {exp_id}")
        return None, None

    df_fold = pd.DataFrame(rows).sort_values("fold")
    fold_results_path = exp_run_dir(exp_id) / f"fold_results_{exp_id}.csv"
    save_csv_atomic(df_fold, fold_results_path)

    metric_cols = ["test_top1", "test_top5", "test_top10", "test_loss", "best_val_top1", "best_val_loss", "best_epoch"]
    summary_rows = []
    for col in metric_cols:
        if col not in df_fold.columns:
            continue
        vals = pd.to_numeric(df_fold[col], errors="coerce")
        summary_rows.append({
            "exp_id": exp_id,
            "name": EXPERIMENTS[exp_id]["name"],
            "metric": col,
            "mean": vals.mean(),
            "std": vals.std(ddof=1),
            "min": vals.min(),
            "max": vals.max(),
            "n_folds": vals.notna().sum(),
        })
    df_summary = pd.DataFrame(summary_rows)
    summary_path = exp_run_dir(exp_id) / f"summary_{exp_id}.csv"
    save_csv_atomic(df_summary, summary_path)

    print("Saved:", fold_results_path)
    print("Saved:", summary_path)
    display(df_fold)
    display(df_summary)
    return df_fold, df_summary

if AUTO_SUMMARIZE_EXPERIMENT:
    for exp_id in ACTIVE_EXPERIMENTS:
        summarize_experiment(exp_id)
else:
    print("AUTO_SUMMARIZE_EXPERIMENT=False")


## 18. Ringkasan Single-Seed

Bagian ini menghitung mean, standard deviation antar-fold, serta perubahan terhadap E0 untuk output dalam satu seed. Agregasi final antar-seed dilakukan setelah seed 42, 123, dan 456 selesai.


In [ ]:

# ============================================================
# 18. ANALISIS BAB 4: MEAN, STD, DELTA E0, RELATIVE CHANGE
# ============================================================


def collect_summary_all_experiments(require_complete=True):
    rows = []
    for exp_id, cfg in EXPERIMENTS.items():
        fold_path = exp_run_dir(exp_id) / f"fold_results_{exp_id}.csv"
        if not fold_path.exists():
            if require_complete:
                print(f"[WAIT] {exp_id} belum punya fold_results_{exp_id}.csv")
                return None
            continue
        df = pd.read_csv(fold_path)
        if require_complete and len(df) < len(TEST_SIGNERS):
            print(f"[WAIT] {exp_id} belum lengkap: {len(df)}/{len(TEST_SIGNERS)} fold")
            return None

        row = {
            "exp_id": exp_id,
            "name": cfg["name"],
            "interp": cfg["interp"],
            "norm": cfg["norm"],
            "augment": cfg["augment"],
            "mean_top1": df["test_top1"].mean(),
            "std_top1": df["test_top1"].std(ddof=1),
            "mean_top5": df["test_top5"].mean(),
            "std_top5": df["test_top5"].std(ddof=1),
            "mean_top10": df["test_top10"].mean(),
            "std_top10": df["test_top10"].std(ddof=1),
        }
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    if "E0" in set(out["exp_id"]):
        e0 = out.loc[out["exp_id"] == "E0", "mean_top1"].iloc[0]
        out["delta_vs_E0"] = out["mean_top1"] - e0
        out["relative_change_vs_E0_pct"] = np.where(abs(e0) > 1e-12, (out["delta_vs_E0"] / e0) * 100.0, np.nan)
    else:
        out["delta_vs_E0"] = np.nan
        out["relative_change_vs_E0_pct"] = np.nan

    order = ["E0", "E1", "E2", "E3", "E4", "E5", "E6", "E7"]
    out["_order"] = out["exp_id"].map({k: i for i, k in enumerate(order)})
    out = out.sort_values("_order").drop(columns="_order")
    return out

summary_preview = collect_summary_all_experiments(require_complete=False)
if summary_preview is not None and not summary_preview.empty:
    display(summary_preview)
else:
    print("Belum ada summary eksperimen yang bisa ditampilkan.")


## 19. Analisis Faktorial 2³

Main effect dihitung sebagai rata-rata performa saat suatu faktor aktif dikurangi rata-rata saat faktor tersebut tidak aktif.


In [ ]:

# ============================================================
# 19. ANALISIS FAKTORIAL 2^3
# ============================================================


def compute_main_effects(summary_all):
    """
    Main effect = rata-rata metrik saat faktor True - rata-rata metrik saat faktor False.
    Desain faktorial 2^3 memakai faktor: interp, norm, augment.
    """
    if summary_all is None or summary_all.empty:
        return pd.DataFrame()

    factors = ["interp", "norm", "augment"]
    metrics = ["mean_top1", "mean_top5", "mean_top10"]
    rows = []
    for factor in factors:
        for metric in metrics:
            true_vals = summary_all[summary_all[factor] == True][metric]
            false_vals = summary_all[summary_all[factor] == False][metric]
            mean_when_on = true_vals.mean()
            mean_when_off = false_vals.mean()
            main_effect = mean_when_on - mean_when_off
            rows.append({
                "factor": factor,
                "metric": metric,
                "mean_when_on": mean_when_on,
                "mean_when_off": mean_when_off,
                "main_effect": main_effect,
                "relative_effect_pct": (main_effect / mean_when_off * 100.0) if abs(mean_when_off) > 1e-12 else np.nan,
                "n_on": int(true_vals.notna().sum()),
                "n_off": int(false_vals.notna().sum()),
            })
    return pd.DataFrame(rows)

if summary_preview is not None and not summary_preview.empty:
    main_effects_preview = compute_main_effects(summary_preview)
    display(main_effects_preview)
else:
    print("Main effects belum bisa dihitung.")


## 20. Export Summary Single-Seed

Summary global dibuat hanya jika seluruh E0–E7 pada seed aktif telah selesai. Output seed lain berada pada folder terpisah.


In [ ]:

# ============================================================
# 20. EXPORT TABEL DAN ARTEFAK BAB 4 + GLOBAL SUMMARY LOCK
# ============================================================

LOCK_TIMEOUT_SECONDS = 600
GLOBAL_LOCK_PATH = RUN_ROOT / "GLOBAL_SUMMARY.lock"


def is_global_summary_locked(lock_path):
    if not lock_path.exists():
        return False
    try:
        lock_time = float(lock_path.read_text().strip())
    except Exception:
        lock_path.unlink(missing_ok=True)
        return False
    if time.time() - lock_time > LOCK_TIMEOUT_SECONDS:
        lock_path.unlink(missing_ok=True)
        return False
    return True


def acquire_global_summary_lock(lock_path):
    if is_global_summary_locked(lock_path):
        return False
    lock_path.write_text(str(time.time()))
    return True


def release_global_summary_lock(lock_path):
    lock_path.unlink(missing_ok=True)


def all_experiments_complete():
    for exp_id in EXPERIMENTS:
        for fold in TEST_SIGNERS:
            if not fold_training_complete(exp_id, fold):
                return False

        fold_path = exp_run_dir(exp_id) / f"fold_results_{exp_id}.csv"
        if not fold_path.exists():
            summarize_experiment(exp_id)

        if not fold_path.exists():
            return False

        df_fold = pd.read_csv(fold_path)
        if len(df_fold) < len(TEST_SIGNERS):
            return False

    return True


def try_write_global_summary():
    if not all_experiments_complete():
        print("[GLOBAL SUMMARY] Belum semua E0-E7 lengkap. Summary global belum dibuat.")
        return None, None

    if not acquire_global_summary_lock(GLOBAL_LOCK_PATH):
        print("[GLOBAL SUMMARY] Lock aktif. Skip agar tidak terjadi race condition.")
        return None, None

    try:
        summary_all = collect_summary_all_experiments(require_complete=True)
        if summary_all is None or summary_all.empty:
            print("[GLOBAL SUMMARY] Tidak ada data summary.")
            return None, None

        main_effects = compute_main_effects(summary_all)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        tmp_summary = RUN_ROOT / f"summary_all_experiments_{timestamp}.tmp.csv"
        final_summary = RUN_ROOT / "summary_all_experiments.csv"
        summary_all.to_csv(tmp_summary, index=False)
        tmp_summary.replace(final_summary)

        tmp_effects = RUN_ROOT / f"main_effects_2x2x2_{timestamp}.tmp.csv"
        final_effects = RUN_ROOT / "main_effects_2x2x2.csv"
        main_effects.to_csv(tmp_effects, index=False)
        tmp_effects.replace(final_effects)

        print("[GLOBAL SUMMARY] Saved:", final_summary)
        print("[GLOBAL SUMMARY] Saved:", final_effects)
        display(summary_all)
        display(main_effects)
        return summary_all, main_effects
    finally:
        release_global_summary_lock(GLOBAL_LOCK_PATH)

if AUTO_TRY_GLOBAL_SUMMARY:
    try_write_global_summary()
else:
    print("AUTO_TRY_GLOBAL_SUMMARY=False")


# 21. Catatan Reproducibility dan Batasan

1. E0–E7 mempertahankan desain faktorial dari interpolasi, normalisasi spasial, dan augmentasi.
2. Interpolasi mengisi missing keypoint hanya pada bagian tengah sequence yang memiliki referensi valid sebelum dan sesudahnya.
3. Normalisasi menggeser pose terhadap pusat tubuh dan menskalakan berdasarkan rata-rata jarak bahu.
4. Standardisasi temporal menjadi 64 frame aktif pada seluruh konfigurasi.
5. Augmentasi hanya diterapkan pada `train_fit`; `val_internal` dan `test` tidak diaugmentasi.
6. LOSO menggunakan signer 1–4 sebagai unseen test signer; test signer tidak masuk train maupun validasi internal.
7. Model terbaik dipilih berdasarkan `validation loss`.
8. Output dipisahkan ke `runs/loso_e0_e7_multiseed/seed_<RUN_SEED>/`.
9. Seed final penelitian adalah 42, 123, dan 456. Menjalankan ketiga seed dengan E0–E7 menghasilkan 3 × 8 × 4 = 96 training fold.
10. `STRICT_DETERMINISM=False` mempertahankan perilaku eksperimen asli. Karena backend CUDA/GPU dapat memiliki nondeterminisme numerik, rerun pada hardware/runtime berbeda tidak dijanjikan bitwise-identical meskipun konfigurasi dan seed sama.
11. Setiap run menyimpan `environment.json`, `study_run_config.json`, konfigurasi preprocessing, checkpoint, log, prediksi test, classification report, dan confusion matrix sebagai provenance.
